In [1152]:
import warnings
warnings.filterwarnings('ignore')

In [1153]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import ast
from sklearn.cluster import KMeans

from rapidfuzz import fuzz, process
import unicodedata

import joblib
import re

In [1154]:
pd.set_option('display.max_columns', 100)

In [1155]:
us_state_to_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "Virgin Islands, U.S.": "VI",
} 

abbrev_to_us_state = dict(map(reversed, us_state_to_abbrev.items()))
# https://gist.github.com/rogerallen/1583593

In [1156]:
# Source: Wikipedia, FEC
# Links:
# https://en.wikipedia.org/wiki/2024_United_States_presidential_election#Electoral_results
# https://www.fec.gov/resources/cms-content/documents/federalelections2020.pdf
# https://www.fec.gov/resources/cms-content/documents/federalelections2016.pdf#page=10
# https://www.fec.gov/resources/cms-content/documents/federalelections2012.pdf#page=11
dem_2pv_24 = 75017613 / (75017613 + 77302580) * 100
dem_2pv_20 = 81283501 / (81283501 + 74223975) * 100
dem_2pv_16 = 65853514 / (65853514 + 62984828) * 100
dem_2pv_12 = 65915795 / (65915795 + 60933504) * 100
dem_2pv_08 = 69498516 / (69498516 + 59948323) * 100

pop_vote = {
    '2008': dem_2pv_08,
    '2012': dem_2pv_12,
    '2016': dem_2pv_16,
    '2020': dem_2pv_20,
    '2024': dem_2pv_24
}

# dem_2pv_08, dem_2pv_12, dem_2pv_16, dem_2pv_20, dem_2pv_24
pop_vote

{'2008': 53.68884751214358,
 '2012': 51.96386225200976,
 '2016': 51.113288930712876,
 '2020': 52.26983492420647,
 '2024': 49.24994613156773}

In [1157]:
data = pd.read_csv('transformed/past_senate_results.csv')
data.head()

,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct
0,2014,Alabama,AL,False,NaN,795606.0,818090.0,[],Jeff Sessions,[],True,[],381899.0,NaN,NaN,NaN,0.0,381899.00,381899.00,0.000000,100.000000
1,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00,11986522.00,50.224502,49.775498
2,2014,Arkansas,AR,False,334174.0,478819.0,847505.0,Mark L. Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11,18893108.11,42.346680,57.653320
3,2014,Colorado,CO,False,944203.0,983891.0,2041058.0,Mark Udall,Cory Gardner,True,False,13414189.0,8725449.0,1928094.0,48.970797,51.029203,13414189.0,8725449.00,22139638.00,60.589017,39.410983
4,2014,Delaware,DE,False,130655.0,98823.0,234038.0,Christopher A. Coons,Kevin Wade,True,False,2845059.0,92155.0,229478.0,56.935741,43.064259,2845059.0,92155.00,2937214.00,96.862503,3.137497


In [1158]:
demo22_cols = ['state', 'white_pct', 'black_pct', 'hisp_pct', 'asn_pct',
                           'natam_pct', 'pi_pct', 'vap_white_pct', 'vap_black_pct', 'vap_hisp_pct', 'vap_asn_pct',
                           'vap_natam_pct', 'vap_pi_pct', 'cit_white_pct', 'cit_black_pct', 'cit_hisp_pct', 'cit_asn_pct',
                           'cit_natam_pct', 'cit_pi_pct', 'cvap_white_pct', 'cvap_black_pct', 'cvap_hisp_pct', 'cvap_asn_pct',
                           'cvap_natam_pct', 'cvap_pi_pct', 'tot_population', 'white_pop', 'black_pop', 'hisp_pop',
                           'asn_pop', 'natam_pop', 'pi_pop', 'vap_pop', 'white_vap_pop', 'black_vap_pop', 'hisp_vap_pop', 'asn_vap_pop',
                           'natam_vap_pop', 'pi_vap_pop', 'cit_pop', 'white_cit_pop', 'black_cit_pop', 'hisp_cit_pop', 'asn_cit_pop',
                           'natam_cit_pop', 'pi_cit_pop', 'cvap_pop', 'white_cvap_pop', 'black_cvap_pop', 'hisp_cvap_pop', 'asn_cvap_pop',
                           'natam_cvap_pop', 'pi_cvap_pop']

demo_cols = ['state', 'white_pct', 'black_pct', 'hisp_pct', 'aapi_pct',
                           'natam_pct', 'other_pct', 'vap_white_pct', 'vap_black_pct', 'vap_hisp_pct', 'vap_aapi_pct',
                           'vap_natam_pct', 'vap_other_pct', 'cit_white_pct', 'cit_black_pct', 'cit_hisp_pct', 'cit_aapi_pct',
                           'cit_natam_pct', 'cit_other_pct', 'cvap_white_pct', 'cvap_black_pct', 'cvap_hisp_pct', 'cvap_aapi_pct',
                           'cvap_natam_pct', 'cvap_other_pct', 'tot_population', 'white_pop', 'black_pop', 'hisp_pop',
                           'aapi_pop', 'natam_pop', 'other_pop', 'vap_pop', 'white_vap_pop', 'black_vap_pop', 'hisp_vap_pop', 'aapi_vap_pop',
                           'natam_vap_pop', 'other_vap_pop', 'cit_pop', 'white_cit_pop', 'black_cit_pop', 'hisp_cit_pop', 'aapi_cit_pop',
                           'natam_cit_pop', 'other_cit_pop', 'cvap_pop', 'white_cvap_pop', 'black_cvap_pop', 'hisp_cvap_pop', 'aapi_cvap_pop',
                           'natam_cvap_pop', 'other_cvap_pop']

In [1159]:
filepath = f'data/demo/2014_114_demo_data_senate/ACSDT5Y2014.B05003D-Data.csv'
demo_b = pd.read_csv(filepath)
demo_b.iloc[0, :].values

array(['Geography', 'Geographic Area Name', 'Estimate!!Total',
       'Margin of Error!!Total', 'Estimate!!Total!!Male',
       'Margin of Error!!Total!!Male',
       'Estimate!!Total!!Male!!Under 18 years',
       'Margin of Error!!Total!!Male!!Under 18 years',
       'Estimate!!Total!!Male!!Under 18 years!!Native',
       'Margin of Error!!Total!!Male!!Under 18 years!!Native',
       'Estimate!!Total!!Male!!Under 18 years!!Foreign born',
       'Margin of Error!!Total!!Male!!Under 18 years!!Foreign born',
       'Estimate!!Total!!Male!!Under 18 years!!Foreign born!!Naturalized U.S. citizen',
       'Margin of Error!!Total!!Male!!Under 18 years!!Foreign born!!Naturalized U.S. citizen',
       'Estimate!!Total!!Male!!Under 18 years!!Foreign born!!Not a U.S. citizen',
       'Margin of Error!!Total!!Male!!Under 18 years!!Foreign born!!Not a U.S. citizen',
       'Estimate!!Total!!Male!!18 years and over',
       'Margin of Error!!Total!!Male!!18 years and over',
       'Estimate!!Total!

In [1160]:
demo_b.iloc[0, :].values[18], demo_b.iloc[0, :].values[40] 

('Estimate!!Total!!Male!!18 years and over!!Native',
 'Estimate!!Total!!Female!!18 years and over!!Native')

In [1161]:
demo_key = {
    '': 'tot',
    'B': 'black',
    'C': 'natam',
    'D': 'asn',
    'E': 'pi',
    'F': 'other_only',
    'G': 'multi',
    'H': 'white',
    'I': 'hisp'
}

demo_14 = pd.DataFrame()

for char in demo_key.keys():
    filepath = f'data/demo/2014_114_demo_data_senate/ACSDT5Y2014.B05003{char}-Data.csv'
    demo_b = pd.read_csv(filepath)
    demo_b_ind = demo_b.iloc[1:, 0:2]
    demo_b_native_male = demo_b.iloc[1:, 18]
    demo_b_nonnat_cit_male = demo_b.iloc[1:, 22]
    demo_b_native_female = demo_b.iloc[1:, 40]
    demo_b_nonnat_cit_female = demo_b.iloc[1:, 44]
    if demo_14.empty:
        demo_b = pd.concat([demo_b_ind, demo_b_native_male, demo_b_nonnat_cit_male, demo_b_native_female, demo_b_nonnat_cit_female], axis=1)
        demo_b = demo_b.set_axis(['geoid', 'state', f'cvap_{demo_key[char]}_native_pop_M', f'cvap_{demo_key[char]}_foreign_pop_M',
                                 f'cvap_{demo_key[char]}_native_pop_F', f'cvap_{demo_key[char]}_foreign_pop_F'], axis=1)
        demo_14 = demo_b.copy()
    else:
        demo_b = pd.concat([demo_b_native_male, demo_b_nonnat_cit_male, demo_b_native_female, demo_b_nonnat_cit_female], axis=1)
        demo_b = demo_b.set_axis([f'cvap_{demo_key[char]}_native_pop_M', f'cvap_{demo_key[char]}_foreign_pop_M',
                                 f'cvap_{demo_key[char]}_native_pop_F', f'cvap_{demo_key[char]}_foreign_pop_F'], axis=1)
        demo_14 = pd.concat([demo_14, demo_b], axis=1)

for race in demo_key.values():
    for sex in ['M', 'F']:
        demo_14[f'cvap_{race}_native_pop_{sex}'] = demo_14[f'cvap_{race}_native_pop_{sex}'].astype(int)
        demo_14[f'cvap_{race}_foreign_pop_{sex}'] = demo_14[f'cvap_{race}_foreign_pop_{sex}'].astype(int)
    demo_14[f'cvap_{race}_pop'] = (demo_14[f'cvap_{race}_native_pop_M'] + demo_14[f'cvap_{race}_foreign_pop_M'] +
                                   demo_14[f'cvap_{race}_native_pop_F'] + demo_14[f'cvap_{race}_foreign_pop_F'])
    if race != 'tot':
        demo_14[f'cvap_{race}_pct'] = demo_14[f'cvap_{race}_pop'] / demo_14['cvap_tot_pop'] * 100

demo_14['cvap_aapi_pct'] = demo_14['cvap_asn_pct'] + demo_14['cvap_pi_pct']
demo_14['cvap_other_pct'] = demo_14['cvap_other_only_pct'] + demo_14['cvap_multi_pct']
#demo_14[['seat_number', 'state']] = demo_14['district_name'].str.extract(r"Congressional District (\d+|\(at Large\)) \(114th Congress\), ([A-za-z\-\s]+)")
#demo_14 = demo_14[~demo_14['seat_number'].isna()]
#demo_14['district'] = demo_14['state'].map(us_state_to_abbrev) + '-' + demo_14['seat_number'].map(lambda x: '00' if (isinstance(x, str) and x == '(at Large)') else
#                                                                                                 (f'0{x}' if int(x) < 10 else str(x)))
demo_14.head()

,geoid,state,cvap_tot_native_pop_M,cvap_tot_foreign_pop_M,cvap_tot_native_pop_F,cvap_tot_foreign_pop_F,cvap_black_native_pop_M,cvap_black_foreign_pop_M,cvap_black_native_pop_F,cvap_black_foreign_pop_F,cvap_natam_native_pop_M,cvap_natam_foreign_pop_M,cvap_natam_native_pop_F,cvap_natam_foreign_pop_F,cvap_asn_native_pop_M,cvap_asn_foreign_pop_M,cvap_asn_native_pop_F,cvap_asn_foreign_pop_F,cvap_pi_native_pop_M,cvap_pi_foreign_pop_M,cvap_pi_native_pop_F,cvap_pi_foreign_pop_F,cvap_other_only_native_pop_M,cvap_other_only_foreign_pop_M,cvap_other_only_native_pop_F,cvap_other_only_foreign_pop_F,cvap_multi_native_pop_M,cvap_multi_foreign_pop_M,cvap_multi_native_pop_F,cvap_multi_foreign_pop_F,cvap_white_native_pop_M,cvap_white_foreign_pop_M,cvap_white_native_pop_F,cvap_white_foreign_pop_F,cvap_hisp_native_pop_M,cvap_hisp_foreign_pop_M,cvap_hisp_native_pop_F,cvap_hisp_foreign_pop_F,cvap_tot_pop,cvap_black_pop,cvap_black_pct,cvap_natam_pop,cvap_natam_pct,cvap_asn_pop,cvap_asn_pct,cvap_pi_pop,cvap_pi_pct,cvap_other_only_pop,cvap_other_only_pct,cvap_multi_pop,cvap_multi_pct,cvap_white_pop,cvap_white_pct,cvap_hisp_pop,cvap_hisp_pct,cvap_aapi_pct,cvap_other_pct
1,0400000US01,Alabama,1686585,23461,1861966,28123,415592,2190,508133,2300,8913,25,9847,69,3461,7692,3209,10639,587,30,392,61,4960,2381,4930,1914,18411,701,18331,634,1218646,6073,1302449,8368,23424,7008,21802,6224,3600135,928215,25.782783,18854,0.523703,25001,0.694446,1070,0.029721,14185,0.394013,38077,1.057655,2535536,70.428914,58458,1.623772,0.724167,1.451668
2,0400000US02,Alaska,261046,12215,231546,14209,9532,826,6695,306,34732,95,34970,93,3279,5919,3474,7830,1946,190,2148,146,1940,727,1651,775,14177,483,14386,464,188639,2801,162006,3574,11237,2054,9990,2087,519016,17359,3.344598,69890,13.465866,20502,3.950167,4430,0.853538,5093,0.981280,29510,5.685759,357020,68.787860,25368,4.887711,4.803705,6.667039
3,0400000US04,Arizona,2034128,149860,2081110,179138,91645,5076,83422,4314,93139,849,101899,779,16823,27777,15071,39908,3938,266,3018,295,82072,18760,76803,20404,41703,3518,43357,4825,1435911,38441,1482657,47185,374754,76261,375442,85222,4444236,184457,4.150477,196666,4.425193,99579,2.240633,7517,0.169140,198039,4.456086,93403,2.101666,3004194,67.597535,911679,20.513740,2.409773,6.557753
4,0400000US05,Arkansas,1020446,18241,1093556,20101,149273,385,172999,347,6351,145,6832,117,2527,4537,2721,6671,346,30,410,72,5807,3210,5421,2424,14508,503,15247,513,826909,3453,875899,5061,22267,9516,20689,7537,2152344,323004,15.007081,13445,0.624668,16456,0.764562,858,0.039864,16862,0.783425,30771,1.429651,1711322,79.509688,60009,2.788077,0.804425,2.213076
5,0400000US06,California,9468906,2212146,9596988,2603248,771274,40938,811590,41376,89305,6037,90535,6762,507635,928134,496270,1145538,37827,8637,38900,9829,722354,296616,704529,329264,363508,58374,376688,63318,5549893,388526,5598679,453459,2328536,813166,2365881,916638,23881288,1665178,6.972731,192639,0.806652,3077577,12.886981,95193,0.398609,2052763,8.595696,861888,3.609052,11990557,50.209005,6424221,26.900647,13.285590,12.204748


In [1162]:
demo_16 = pd.read_csv('data/demo/2016_115_acs_demo_states.csv')
demo_16 = demo_16.iloc[2:]
demo_16 = demo_16.set_axis(demo_cols, axis=1)
demo_16.head()

,state,white_pct,black_pct,hisp_pct,aapi_pct,natam_pct,other_pct,vap_white_pct,vap_black_pct,vap_hisp_pct,vap_aapi_pct,vap_natam_pct,vap_other_pct,cit_white_pct,cit_black_pct,cit_hisp_pct,cit_aapi_pct,cit_natam_pct,cit_other_pct,cvap_white_pct,cvap_black_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_other_pct,tot_population,white_pop,black_pop,hisp_pop,aapi_pop,natam_pop,other_pop,vap_pop,white_vap_pop,black_vap_pop,hisp_vap_pop,aapi_vap_pop,natam_vap_pop,other_vap_pop,cit_pop,white_cit_pop,black_cit_pop,hisp_cit_pop,aapi_cit_pop,natam_cit_pop,other_cit_pop,cvap_pop,white_cvap_pop,black_cvap_pop,hisp_cvap_pop,aapi_cvap_pop,natam_cvap_pop,other_cvap_pop
2,Alabama,66.3,26.4,4.0,1.3,0.5,1.6,68.5,25.5,3.2,1.3,0.5,1.1,67.5,26.9,2.8,0.8,0.5,1.6,70.0,26.0,1.7,0.8,0.5,1.1,"4,841,160","3,208,550","1,278,745","193,500","63,040","21,960","75,345","3,735,980","2,558,825","951,850","118,885","48,080","17,545","40,785","4,734,505","3,195,135","1,271,600","131,865","39,820","21,910","74,180","3,639,495","2,546,550","945,510","62,375","27,795","17,500",39735
3,Alaska,62.1,3.2,6.7,7.1,13.7,7.3,66.1,3.2,5.8,7.0,12.5,5.4,63.5,3.1,6.4,5.4,14.1,7.5,67.9,3.1,5.4,5.0,13.0,5.6,"736,855","457,540","23,235","49,030","52,365","100,715","53,970","549,240","363,045","17,565","31,785","38,660","68,595","29,590","712,745","452,800","21,855","45,260","38,585","100,690","53,555","527,810","358,490","16,520","28,280","26,625","68,570",29325
4,Arizona,56.3,4.0,30.5,3.2,4.0,2.0,61.3,3.9,26.4,3.3,3.6,1.4,60.0,4.2,27.0,2.4,4.3,2.1,66.4,4.1,21.5,2.5,4.0,1.5,"6,728,580","3,786,195","271,740","2,054,850","213,570","266,975","135,245","5,108,965","3,131,220","200,910","1,348,580","169,235","185,970","73,060","6,192,380","3,713,075","257,705","1,671,880","150,875","266,305","132,560","4,613,575","3,062,780","190,080","991,500","113,155","185,380",70685
5,Arkansas,73.5,15.5,7.0,1.6,0.5,1.9,76.3,14.6,5.5,1.6,0.6,1.4,75.6,15.9,5.0,1.1,0.6,2.0,78.9,15.1,3.0,0.9,0.6,1.4,"2,968,470","2,180,390","458,875","207,055","48,875","16,325","56,980","2,261,240","1,725,465","330,180","125,495","35,935","12,815","31,365","2,874,015","2,171,525","456,850","142,530","30,670","16,295","56,150","2,175,340","1,717,230","328,360","66,205","20,095","12,790",30665
6,California,38.6,5.6,38.6,14.1,0.4,2.7,42.3,5.8,34.4,15.0,0.4,2.1,43.2,6.3,34.2,12.8,0.4,3.1,48.9,6.7,28.0,13.5,0.4,2.4,"38,654,210","14,906,720","2,179,645","14,903,970","5,467,440","138,565","1,057,885","29,513,915","12,484,440","1,700,180","10,166,495","4,435,330","109,655","617,770","33,355,855","14,398,410","2,117,530","11,422,245","4,256,750","137,295","1,023,655","24,582,595","12,021,740","1,644,655","6,891,460","3,327,435","108,530",588785


In [1163]:
demo_18 = pd.read_csv('data/demo/2018_116_acs_demo_states.csv')
demo_18 = demo_18.iloc[2:]
demo_18 = demo_18.set_axis(demo_cols, axis=1)
demo_18.head()

,state,white_pct,black_pct,hisp_pct,aapi_pct,natam_pct,other_pct,vap_white_pct,vap_black_pct,vap_hisp_pct,vap_aapi_pct,vap_natam_pct,vap_other_pct,cit_white_pct,cit_black_pct,cit_hisp_pct,cit_aapi_pct,cit_natam_pct,cit_other_pct,cvap_white_pct,cvap_black_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_other_pct,tot_population,white_pop,black_pop,hisp_pop,aapi_pop,natam_pop,other_pop,vap_pop,white_vap_pop,black_vap_pop,hisp_vap_pop,aapi_vap_pop,natam_vap_pop,other_vap_pop,cit_pop,white_cit_pop,black_cit_pop,hisp_cit_pop,aapi_cit_pop,natam_cit_pop,other_cit_pop,cvap_pop,white_cvap_pop,black_cvap_pop,hisp_cvap_pop,aapi_cvap_pop,natam_cvap_pop,other_cvap_pop
2,Alabama,65.8,26.5,4.2,1.4,0.5,1.7,68.0,25.7,3.3,1.4,0.5,1.2,67.0,26.9,3.0,0.9,0.5,1.7,69.4,26.2,1.9,0.8,0.5,1.2,"4,864,680","3,201,535","1,289,275","203,150","66,755","23,275","80,670","3,765,885","2,561,630","966,735","123,145","51,340","18,800","44,210","4,759,650","3,187,675","1,281,805","144,760","42,480","23,240","79,695","3,671,105","2,548,845","960,070","70,240","29,865","18,765",43305
3,Alaska,61.2,3.1,6.9,7.4,14.0,7.3,65.1,3.2,6.0,7.4,12.8,5.4,62.7,3.0,6.6,5.7,14.5,7.5,67.0,3.1,5.6,5.4,13.3,5.5,"738,515","452,105","22,995","51,185","54,490","103,610","54,130","552,380","359,875","17,625","33,375","40,935","70,850","29,720","713,900","447,265","21,300","47,320","40,705","103,540","53,770","530,385","355,115","16,430","29,915","28,725","70,780",29425
4,Arizona,55.2,4.2,31.1,3.4,3.9,2.2,60.0,4.1,27.2,3.6,3.6,1.6,58.7,4.3,27.9,2.6,4.2,2.3,64.8,4.2,22.7,2.6,4.0,1.7,"6,946,685","3,834,835","288,555","2,163,305","237,485","272,475","150,020","5,312,900","3,188,280","215,405","1,443,570","190,780","192,165","82,680","6,408,360","3,761,305","273,800","1,790,350","163,700","271,705","147,495","4,812,765","3,119,860","204,230","1,092,105","124,550","191,560",80450
5,Arkansas,72.8,15.4,7.3,1.8,0.6,2.2,75.6,14.6,5.9,1.7,0.6,1.6,74.9,15.8,5.4,1.1,0.6,2.2,78.3,15.1,3.4,1.0,0.6,1.6,"2,990,670","2,176,835","459,940","219,055","52,785","17,365","64,700","2,284,725","1,728,370","333,360","133,830","39,345","13,445","36,385","2,893,855","2,168,105","457,480","155,000","32,160","17,365","63,750","2,195,865","1,720,325","331,070","74,000","21,410","13,445",35625
6,California,37.7,5.6,38.9,14.6,0.4,2.9,41.2,5.7,34.9,15.5,0.4,2.2,42.0,6.3,35.1,13.1,0.4,3.2,47.3,6.6,29.2,13.9,0.4,2.5,"39,148,750","14,768,660","2,187,355","15,221,585","5,712,110","140,785","1,118,215","30,075,110","12,402,810","1,724,500","10,509,560","4,668,255","111,145","658,840","33,964,450","14,256,210","2,124,855","11,911,545","4,452,755","139,380","1,079,705","25,232,630","11,940,360","1,668,855","7,374,130","3,511,600","109,940",627735


In [1164]:
demo_20 = pd.read_csv('data/demo/2019_117_acs_demo_states.csv')
demo_20 = demo_20.iloc[2:]
demo_20 = pd.concat([demo_20.iloc[:, 0], demo_20.iloc[:, 19:25]], axis=1)
demo_20 = demo_20.set_axis(['state', 'cvap_white_pct', 'cvap_black_pct', 'cvap_hisp_pct', 'cvap_aapi_pct', 'cvap_natam_pct', 'cvap_other_pct'], axis=1)
demo_20.head()

,state,cvap_white_pct,cvap_black_pct,cvap_hisp_pct,cvap_aapi_pct,cvap_natam_pct,cvap_other_pct
2,Alabama,69.2,26.3,2.0,0.9,0.5,1.2
3,Alaska,66.6,3.1,5.7,5.4,13.7,5.5
4,Arizona,64.2,4.3,23.2,2.6,4.0,1.7
5,Arkansas,78.0,15.1,3.6,1.0,0.6,1.7
6,California,46.6,6.6,29.8,14.1,0.4,2.5


In [1165]:
demo_22 = pd.read_csv('data/demo/2022_118_acs_demo_states.csv')
demo_22 = demo_22.iloc[2:]
demo_22 = demo_22.set_axis(demo22_cols, axis=1)
demo_22['cvap_aapi_pct'] = demo_22['cvap_asn_pct'].astype(float) + demo_22['cvap_pi_pct'].astype(float) # Combine Asian and Pacific Islander percentages for consistency
demo_22.head()

,state,white_pct,black_pct,hisp_pct,asn_pct,natam_pct,pi_pct,vap_white_pct,vap_black_pct,vap_hisp_pct,vap_asn_pct,vap_natam_pct,vap_pi_pct,cit_white_pct,cit_black_pct,cit_hisp_pct,cit_asn_pct,cit_natam_pct,cit_pi_pct,cvap_white_pct,cvap_black_pct,cvap_hisp_pct,cvap_asn_pct,cvap_natam_pct,cvap_pi_pct,tot_population,white_pop,black_pop,hisp_pop,asn_pop,natam_pop,pi_pop,vap_pop,white_vap_pop,black_vap_pop,hisp_vap_pop,asn_vap_pop,natam_vap_pop,pi_vap_pop,cit_pop,white_cit_pop,black_cit_pop,hisp_cit_pop,asn_cit_pop,natam_cit_pop,pi_cit_pop,cvap_pop,white_cvap_pop,black_cvap_pop,hisp_cvap_pop,asn_cvap_pop,natam_cvap_pop,pi_cvap_pop,cvap_aapi_pct
2,Alabama,65.1,27.3,4.6,1.7,1.1,0.0,67.3,26.2,3.6,1.7,1.2,0.0,66.1,27.7,3.5,1.3,1.1,0.0,68.6,26.7,2.3,1.2,1.2,0.0,"5,028,090","3,271,225","1,370,580","232,405","87,785","56,240","1,590","3,917,450","2,634,490","1,024,660","141,250","65,380","46,045","1,260","4,924,860","3,257,200","1,363,930","173,605","64,645","56,185","1,475","3,824,040","2,621,730","1,019,225","87,570","44,400","45,995","1,160",1.2
3,Alaska,59.3,4.2,7.5,7.7,18.5,1.5,63.3,3.9,6.5,7.5,16.5,1.3,60.5,4.2,7.2,6.4,19.1,1.4,64.8,3.9,6.2,5.9,17.1,1.1,"734,820","436,030","30,980","54,890","56,615","135,985","10,950","555,485","351,895","21,915","36,265","41,870","91,820","7,020","712,100","430,570","29,850","51,190","45,555","135,855","9,795","534,725","346,615","21,035","33,030","31,740","91,685","6,005",7.0
4,Arizona,53.7,5.3,32.0,4.1,4.3,0.2,58.1,4.9,28.4,4.1,4.0,0.2,56.7,5.5,29.3,3.4,4.6,0.2,62.2,5.1,24.6,3.2,4.4,0.2,"7,172,280","3,850,230","381,250","2,297,515","297,185","307,960","12,925","5,578,820","3,243,070","272,780","1,584,280","228,895","224,660","10,200","6,681,080","3,789,020","366,220","1,955,930","226,140","307,595","11,615","5,118,555","3,185,580","261,015","1,259,165","165,105","224,300","9,090",3.4
5,Arkansas,71.5,16.1,8.1,1.9,1.8,0.4,74.3,15.0,6.6,1.8,1.8,0.3,73.6,16.6,6.1,1.4,1.8,0.2,76.9,15.5,4.3,1.3,1.8,0.1,"3,018,670","2,159,400","487,120","243,320","58,135","53,750","11,090","2,321,400","1,725,495","348,780","153,840","42,315","41,145","7,035","2,920,760","2,150,710","484,320","179,480","42,220","53,745","4,755","2,233,470","1,717,850","346,310","94,935","28,600","41,140","2,120",1.4
6,California,35.8,6.2,39.7,16.5,0.8,0.4,39.0,6.2,36.2,16.8,0.9,0.4,39.4,6.9,36.6,15.1,1.0,0.4,44.0,7.1,31.7,15.3,1.0,0.4,"39,356,105","14,099,515","2,450,320","15,617,930","6,489,585","332,895","141,855","30,581,535","11,928,785","1,904,520","11,084,740","5,151,220","265,030","113,270","34,550,270","13,609,700","2,379,290","12,662,405","5,227,415","330,580","123,850","26,078,140","11,485,705","1,841,855","8,266,065","3,997,305","262,900","96,705",15.7


In [1166]:
demo_24 = pd.read_csv('data/demo/2024_119_acs_demo_census.csv') # This is CVAP only
demo_24 = pd.pivot_table(demo_24, columns=['lntitle'], index=['geoname', 'geoid'], values='cvap_est', aggfunc='first')
demo_24 = demo_24.reset_index()
extr_tokens = demo_24['geoname'].str.extract(r"Congressional District (\(at Large\)|\d+) \(119th Congress\), ([A-za-z\s]+)")
demo_24 = pd.concat([demo_24, extr_tokens], axis=1)
demo_24 = demo_24.rename({0: 'seat_number', 1: 'state'}, axis=1)
demo_24 = demo_24[~demo_24['state'].isna()]
demo_24['district'] = demo_24['state'].astype(str).map(us_state_to_abbrev) + '-' + demo_24['seat_number'].map(lambda x: '00' if x == '(at Large)' else 
                                                                                                              (f'0{x}' if int(x) < 10 else f'{x}'))
demo_24 = demo_24.set_axis(['geoname', 'geoid', 'natam', 'natamXblack', 'natamXwhite', 'asn', 'asnXwhite', 'black', 'blackXwhite',
                           'hisp', 'pi', 'nothisp', 'multi', 'tot', 'white', 'seat_number', 'state', 'district'], axis=1)
demo_24['aapi'] = demo_24['asn'].astype(int) + demo_24['pi'].astype(int)
demo_24 = demo_24[['state', 'natam', 'aapi', 'black', 'hisp', 'white', 'pi', 'tot']].groupby(['state']).sum()
for race in ['natam', 'aapi', 'black', 'hisp', 'white', 'pi']:
    demo_24[f'cvap_{race}_pct'] = demo_24[race] / demo_24['tot'] * 100
demo_24 = demo_24.rename({'tot': 'cvap_pop'}, axis=1)
demo_24 = demo_24.reset_index()
demo_24.head()

,state,natam,aapi,black,hisp,white,pi,cvap_pop,cvap_natam_pct,cvap_aapi_pct,cvap_black_pct,cvap_hisp_pct,cvap_white_pct,cvap_pi_pct
0,Alabama,11333,39431,989960,106822,2614645,1377,3845692,0.294693,1.025329,25.742051,2.777706,67.988934,0.035806
1,Alaska,68109,34923,16582,32606,347344,6266,538819,12.640423,6.481397,3.077471,6.051383,64.463948,1.162914
2,Arizona,177222,155799,236999,1296211,3296328,9045,5324108,3.328670,2.926293,4.451431,24.346069,61.913244,0.169888
3,Arkansas,8361,26690,330230,110187,1709182,2421,2245935,0.372273,1.188369,14.703453,4.906064,76.101134,0.107795
4,California,83078,3960026,1640805,8458349,11147879,96710,26132787,0.317907,15.153478,6.278722,32.366808,42.658592,0.370072


In [1167]:
# pd.set_option('display.max_columns', 300)
educinc_2014 = pd.read_csv('data/demo/ACSST5Y2014.S1501-Data-STATES.csv')
educinc_2014 = educinc_2014.set_axis(educinc_2014.iloc[0], axis=1).iloc[1:]
# #educinc_2014 = educinc_2014[['Geography', 'Geographic Area Name', "Total!!Estimate!!Bachelor's degree or higher"]]
educinc_2014.head(2)

,Geography,Geographic Area Name,Total!!Estimate!!Population 18 to 24 years,Total!!Margin of Error!!Population 18 to 24 years,Male!!Estimate!!Population 18 to 24 years,Male!!Margin of Error!!Population 18 to 24 years,Female!!Estimate!!Population 18 to 24 years,Female!!Margin of Error!!Population 18 to 24 years,Total!!Estimate!!Less than high school graduate,Total!!Margin of Error!!Less than high school graduate,Male!!Estimate!!Less than high school graduate,Male!!Margin of Error!!Less than high school graduate,Female!!Estimate!!Less than high school graduate,Female!!Margin of Error!!Less than high school graduate,Total!!Estimate!!High school graduate (includes equivalency),Total!!Margin of Error!!High school graduate (includes equivalency),Male!!Estimate!!High school graduate (includes equivalency),Male!!Margin of Error!!High school graduate (includes equivalency),Female!!Estimate!!High school graduate (includes equivalency),Female!!Margin of Error!!High school graduate (includes equivalency),Total!!Estimate!!Some college or associate's degree,Total!!Margin of Error!!Some college or associate's degree,Male!!Estimate!!Some college or associate's degree,Male!!Margin of Error!!Some college or associate's degree,Female!!Estimate!!Some college or associate's degree,Female!!Margin of Error!!Some college or associate's degree,Total!!Estimate!!Bachelor's degree or higher,Total!!Margin of Error!!Bachelor's degree or higher,Male!!Estimate!!Bachelor's degree or higher,Male!!Margin of Error!!Bachelor's degree or higher,Female!!Estimate!!Bachelor's degree or higher,Female!!Margin of Error!!Bachelor's degree or higher,Total!!Estimate!!Population 25 years and over,Total!!Margin of Error!!Population 25 years and over,Male!!Estimate!!Population 25 years and over,Male!!Margin of Error!!Population 25 years and over,Female!!Estimate!!Population 25 years and over,Female!!Margin of Error!!Population 25 years and over,Total!!Estimate!!Population 25 years and over!!Less than 9th grade,Total!!Margin of Error!!Population 25 years and over!!Less than 9th grade,Male!!Estimate!!Population 25 years and over!!Less than 9th grade,Male!!Margin of Error!!Population 25 years and over!!Less than 9th grade,Female!!Estimate!!Population 25 years and over!!Less than 9th grade,Female!!Margin of Error!!Population 25 years and over!!Less than 9th grade,"Total!!Estimate!!Population 25 years and over!!9th to 12th grade, no diploma","Total!!Margin of Error!!Population 25 years and over!!9th to 12th grade, no diploma","Male!!Estimate!!Population 25 years and over!!9th to 12th grade, no diploma","Male!!Margin of Error!!Population 25 years and over!!9th to 12th grade, no diploma","Female!!Estimate!!Population 25 years and over!!9th to 12th grade, no diploma","Female!!Margin of Error!!Population 25 years and over!!9th to 12th grade, no diploma",...,Female!!Margin of Error!!POVERTY RATE FOR THE POPULATION 25 YEARS AND OVER FOR WHOM POVERTY STATUS IS DETERMINED BY EDUCATIONAL ATTAINMENT LEVEL!!Some college or associate's degree,Total!!Estimate!!POVERTY RATE FOR THE POPULATION 25 YEARS AND OVER FOR WHOM POVERTY STATUS IS DETERMINED BY EDUCATIONAL ATTAINMENT LEVEL!!Bachelor's degree or higher,Total!!Margin of Error!!POVERTY RATE FOR THE POPULATION 25 YEARS AND OVER FOR WHOM POVERTY STATUS IS DETERMINED BY EDUCATIONAL ATTAINMENT LEVEL!!Bachelor's degree or higher,Male!!Estimate!!POVERTY RATE FOR THE POPULATION 25 YEARS AND OVER FOR WHOM POVERTY STATUS IS DETERMINED BY EDUCATIONAL ATTAINMENT LEVEL!!Bachelor's degree or higher,Male!!Margin of Error!!POVERTY RATE FOR THE POPULATION 25 YEARS AND OVER FOR WHOM POVERTY STATUS IS DETERMINED BY EDUCATIONAL ATTAINMENT LEVEL!!Bachelor's degree or higher,Female!!Estimate!!POVERTY RATE FOR THE POPULATION 25 YEARS AND OVER FOR WHOM POVERTY STATUS IS DETERMINED BY EDUCATIONAL ATTAINMENT LEVEL!!Bachelor's degree or higher,Female!!Margin of Error!!POVERTY RATE FOR THE POPULATION 25 YEARS AND OVER FOR WHOM POVERTY STATUS IS DETERMINED BY EDUCATIONA

In [1168]:
educinc_2014 = pd.read_csv('data/demo/ACSST5Y2014.S1501-Data-STATES.csv')
educinc_2014 = educinc_2014.set_axis(educinc_2014.iloc[0], axis=1).iloc[1:]
educinc_2014 = educinc_2014[['Geography', 'Geographic Area Name', 
                             #"Total!!Estimate!!Population 18 to 24 years",
                            #"Total!!Estimate!!Bachelor's degree or higher", "Total!!Estimate!!Population 25 years and over",
                            "Total!!Estimate!!Percent bachelor's degree or higher"]]
educinc_2014 = educinc_2014.rename({
    "Total!!Estimate!!Percent bachelor's degree or higher": "college"
}, axis=1)
educinc_2014 = educinc_2014.rename({
    'Geographic Area Name': 'State'
}, axis=1)
educinc_2014.head()

,Geography,State,college
1,0400000US01,Alabama,23.1
2,0400000US02,Alaska,27.7
3,0400000US04,Arizona,27.1
4,0400000US05,Arkansas,20.6
5,0400000US06,California,31.0


In [1169]:
educinc_2016 = pd.read_csv('data/demo/ACSST5Y2016.S1501-Data-STATES.csv')
cols = educinc_2016.iloc[0]
#educinc_2016 = educinc_2016[['Geography', 'Geographic Area Name', 'Total!!Estimate']]
bach_plus_cols = np.array([str(s) for s in cols.values if re.search(r"Bachelor's degree or higher", str(s))])
tot_est_cols = np.array([str(s) for s in cols.values if re.search(r"Total\!\!Estimate\!\!", str(s))])
educinc_2016 = educinc_2016.set_axis(cols, axis=1).iloc[1:]
educinc_16_cols = educinc_2016[["Percent!!Estimate!!Percent bachelor's degree or higher"]]
educinc_16_ind = educinc_2016[['Geography', 'Geographic Area Name']]
educinc_2016 = pd.concat([educinc_16_ind, educinc_16_cols], axis=1)
educinc_2016 = educinc_2016.rename({
    "Percent!!Estimate!!Percent bachelor's degree or higher": "college"
}, axis=1)
#educinc_2016[['state', 'seat_number']] = educinc_2016['Geographic Area Name'].str.extract(r"Congressional District (\d+|\(at Large\)) \(115th Congress\), ([A-za-z\-\s]+)")
#educinc_2016 = educinc_2016[~educinc_2016['seat_number'].isna()]
#educinc_2016['District'] = educinc_2016['seat_number'].map(us_state_to_abbrev) + '-' + educinc_2016['state'].map(lambda x: 'AL' if (isinstance(x, str) and x == '(at Large)') else
#                                                                                                 (f'0{x}' if int(x) < 10 else str(x)))
educinc_2016 = educinc_2016.rename({
    'Geographic Area Name': 'State'
}, axis=1)
educinc_2016.head()

,Geography,State,college
1,0400000US01,Alabama,24.0
2,0400000US02,Alaska,28.8
3,0400000US04,Arizona,28.0
4,0400000US05,Arkansas,21.5
5,0400000US06,California,32.0


In [1170]:
educinc_2018 = pd.read_csv('data/demo/2018_116_acs_educ_inc_states.csv')
educinc_2018 = educinc_2018.rename({
    "Bachelor's": "college", # "Bachelor's" column indicates bachelor's or higher
    "Median Income": "median_income"
}, axis=1)
educinc_2018 = educinc_2018.iloc[1:]
educinc_2018['college'] = educinc_2018['college'].astype(float)
educinc_2018['median_income'] = educinc_2018['median_income'].str.lstrip('$').str.replace(',', '').astype(int)
educinc_2018.head()

,State,Clinton,Trump,Obama,Romney,college,Rank,White Bachelor's,Rank.1,median_income,Rank.2,White Income,Rank.3
1,Alabama,34.4,62.1,38.4,60.5,24.9,44,27.6,44,48486,44,"$55,690",42
2,Alaska,36.6,51.3,40.8,54.8,29.2,29,35.6,19,76715,5,"$85,448",4
3,Arizona,44.6,48.1,44.4,53.5,28.9,31,35.1,21,56213,30,"$62,261",26
4,Arkansas,33.7,60.6,36.9,60.6,22.6,48,24.4,48,45726,48,"$49,996",49
5,California,61.5,31.5,60.2,37.1,33.3,13,43.2,6,71228,9,"$82,970",6


In [1171]:
educinc_2020 = pd.read_csv('data/demo/2019_117_acs_educ_inc_states.csv')
educinc_2020 = educinc_2020.rename({
    "Bachelor's": "college", # "Bachelor's" column indicates bachelor's or higher
    "Median Income": "median_income"
}, axis=1)
educinc_2020 = educinc_2020.iloc[1:]
educinc_2020['college'] = educinc_2020['college'].astype(float)
educinc_2020['median_income'] = educinc_2020['median_income'].str.lstrip('$').str.replace(',', '').fillna(0).astype(int)
# We apparently don't have data for median income for North Carolina's 2020 plan
educinc_2020.head()

,State,Biden,Trump,Clinton,Trump.1,Obama,Romney,college,Rank,White Bachelor's,Rank.1,median_income,Rank.2,White Income,Rank.3
1,Alabama,36.6,62.0,34.4,62.1,38.4,60.5,25.5,43,28.1,44,50536,45,"$57,935",42
2,Alaska,42.8,52.8,36.6,51.3,40.8,54.8,29.6,30,36.1,19,77640,6,"$85,841",6
3,Arizona,49.2,48.9,44.6,48.1,44.4,53.5,29.5,31,35.7,21,58945,29,"$64,657",26
4,Arkansas,34.8,62.4,33.7,60.6,36.9,60.6,23.0,48,24.8,48,47597,48,"$51,681",49
5,California,63.4,34.3,61.5,31.5,60.2,37.1,33.9,15,44.0,7,75235,8,"$87,089",5


In [1172]:
educinc_2022 = pd.read_csv('data/demo/2022_118_acs_educ_inc_states.csv')
educinc_2022 = educinc_2022.rename({
    "Bachelor's": "college", # "Bachelor's" column indicates bachelor's or higher
    "Median Income": "median_income"
}, axis=1)
educinc_2022 = educinc_2022.iloc[1:]
educinc_2022['college'] = educinc_2022['college'].astype(float)
educinc_2022['median_income'] = educinc_2022['median_income'].str.lstrip('$').str.replace(',', '').astype(int)
educinc_2022.head()

,State,Biden,Trump,Clinton,Trump.1,college,Rank,White Bachelor's,Rank.1,median_income,Rank.2,White Income,Rank.3
1,Alabama,36.6,62.0,34.4,62.1,27.2,44.0,30.1,44.0,59674,44.0,"$68,212",44.0
2,Alaska,42.8,52.8,36.6,51.3,30.7,33.0,37.3,22.0,88121,11.0,"$96,116",6.0
3,Arizona,49.2,48.9,44.6,48.1,31.8,29.0,38.2,19.0,74568,19.0,"$80,637",20.0
4,Arkansas,34.8,62.4,33.7,60.6,24.7,48.0,26.6,49.0,55432,47.0,"$60,593",49.0
5,California,63.4,34.3,61.5,31.5,35.9,15.0,46.1,7.0,91551,5.0,"$103,065",3.0


In [1173]:
educinc_2024 = pd.read_csv('data/demo/ACSST5Y2024.S1501-Data-STATES.csv')
educinc_2024 = educinc_2024.set_axis(educinc_2024.iloc[0], axis=1).iloc[1:]
educinc_2024 = educinc_2024[['Geography', 'Geographic Area Name', 
                             "Estimate!!Total!!AGE BY EDUCATIONAL ATTAINMENT!!Population 25 years and over!!Bachelor's degree or higher",
                            "Estimate!!Total!!AGE BY EDUCATIONAL ATTAINMENT!!Population 25 years and over"]]
educinc_2024 = educinc_2024.rename({
    "Estimate!!Total!!AGE BY EDUCATIONAL ATTAINMENT!!Population 25 years and over!!Bachelor's degree or higher": "college_25+",
    "Estimate!!Total!!AGE BY EDUCATIONAL ATTAINMENT!!Population 25 years and over": "pop_25+"
}, axis=1)
educinc_2024['college'] = educinc_2024['college_25+'].astype(float) / educinc_2024['pop_25+'].astype(float) * 100
educinc_2024 = educinc_2024.rename({
    'Geographic Area Name': 'State'
}, axis=1)
educinc_2024.head()

,Geography,State,college_25+,pop_25+,college
1,0400000US01,Alabama,985483,3473706,28.369787
2,0400000US02,Alaska,155356,490552,31.669629
3,0400000US04,Arizona,1696340,5088675,33.335593
4,0400000US05,Arkansas,529858,2059172,25.731605
5,0400000US06,California,10039805,27073960,37.082883


In [1174]:
## Polling averages 2014-16
pollavg_1416 = pd.read_csv('transformed/senate_polling_averages_2014-16.csv')
pollavg_1416 = pollavg_1416.rename({
    'year': 'cycle',
    'location': 'state_po'
}, axis=1)
pollavg_1416['state'] = pollavg_1416['state_po'].map(abbrev_to_us_state)
pollavg_1416 = pollavg_1416[['cycle', 'state', 'state_po', 'candidate_name', 'party', 'avg', 'std', 'lower_ci', 'upper_ci', 'effn']]
pollavg_1416.head()

,cycle,state,state_po,candidate_name,party,avg,std,lower_ci,upper_ci,effn
0,2014,Colorado,CO,Mark Udall,DEM,44.062920,2.242446,39.667726,48.458114,9.133357
1,2014,Georgia,GA,Mary Michelle Nunn,DEM,45.051104,1.812236,41.499121,48.603086,8.728368
2,2014,Louisiana,LA,Mary L. Landrieu,DEM,40.362491,4.021863,32.479640,48.245342,5.480770
3,2014,Hawaii,HI,Brian Schatz,DEM,68.328108,5.208750,58.118958,78.537258,1.139416
4,2014,Minnesota,MN,Al Franken,DEM,50.596116,1.519847,47.617217,53.575016,1.939826


In [1175]:
pollavg[(pollavg['cycle'] == 2024) & (pollavg['state'] == 'Maine')]

,cycle,state,candidate_name,party,avg,std,lower_ci,upper_ci,effn,state_po
36,2024,Maine,David Allen Costello,DEM,7.451646,0.499659,6.472314,8.430978,2.334136,NaN
37,2024,Maine,Demi Kouzounas,REP,31.801971,3.567984,24.808723,38.795219,2.334136,NaN
38,2024,Maine,Angus S. King Jr.,IND,51.781937,2.000030,47.861879,55.701995,2.334136,NaN


In [1176]:
# Poll averages
pollavg = pd.read_csv('transformed/senate_polling_averages.csv')
pollavg_1416['state_po'] = pollavg_1416['state'].map(us_state_to_abbrev)
pollavg = pd.concat([pollavg, pollavg_1416], axis=0)
def get_poll_avg(year, state, special, party, candidate):
    # Candidate as noted in past_house_results.csv
    # Party is 'DEM' or 'REP'
    poll_df = pollavg[
        (pollavg['cycle'] == year) &
        (pollavg['state'] == state) &
        (pollavg['party'].isin([party, 'IND', 'OTH']))
    ]
    res_df = data[
        (data['year'] == year) &
        (data['state'] == state) &
        (data['special'] == special)
    ]

    if party == 'DEM':
        res_col = 'dem_cand'
    elif party == 'REP':
        res_col = 'rep_cand'
    else:
        raise ValueError('Not a valid party for this function')

    if res_df.shape[0] == 0:
        return pd.Series({f'{party.lower()}_fuzzymatch': float('nan'),
               f'{party.lower()}_poll_avg': float('nan'),
               f'{party.lower()}_effn': float('nan')})
    if poll_df.shape[0] == 0:
        return pd.Series({f'{party.lower()}_fuzzymatch': float('nan'),
               f'{party.lower()}_poll_avg': float('nan'),
               f'{party.lower()}_effn': float('nan')})

    candidate_in_res = res_df[res_col].values[0]

    if candidate_in_res[0] != '[':
        fuzzymatch = process.extractOne(candidate_in_res, poll_df['candidate_name'].values, scorer=fuzz.WRatio, score_cutoff=55)
        cand_poll_df = poll_df[poll_df['candidate_name'] == fuzzymatch[0]]
        if cand_poll_df.shape[0] > 1:
            print(cand_poll_df['candidate_name'].values[0])
            raise ValueError(f"Something went wrong in the fuzzy match. cycle={year}, state={state}, party={party}.")
        avg, effn = cand_poll_df['avg'].values[0], poll_df['effn'].values[0]
        return pd.Series({f'{party.lower()}_fuzzymatch': float('nan') if fuzzymatch is None else fuzzymatch[0],
               f'{party.lower()}_poll_avg': avg,
               f'{party.lower()}_effn': effn})
    else:
        matches = []
        averages = []
        enops = []
        if len(ast.literal_eval(candidate_in_res)) == 0:
            return pd.Series({f'{party.lower()}_fuzzymatch': float('nan'),
               f'{party.lower()}_poll_avg': float('nan'),
               f'{party.lower()}_effn': float('nan')})
        for cand in ast.literal_eval(candidate_in_res):
            fuzzymatch = process.extractOne(cand, poll_df['candidate_name'].values, scorer=fuzz.WRatio, score_cutoff=55)
            if state == 'Louisiana' and year == 2016 and cand == 'John Fleming':
                fuzzymatch = None
            cand_poll_df = poll_df[poll_df['candidate_name'] == fuzzymatch]
            if cand_poll_df.shape[0] > 1:
                print(cand_poll_df['candidate_name'].values[0])
                raise ValueError(f"Something went wrong in the fuzzy match. cycle={year}, state={state}, party={party}.")
            avg, effn = poll_df['avg'].values[0], poll_df['effn'].values[0]
            matches.append(float('nan') if fuzzymatch is None else fuzzymatch[0])
            averages.append(0 if fuzzymatch is None else avg)
            enops.append(0 if fuzzymatch is None else effn)
        return pd.Series({f'{party.lower()}_fuzzymatch': matches,
               f'{party.lower()}_poll_avg': np.sum(np.array(averages)),
               f'{party.lower()}_effn': max(enops)})

In [1177]:
data[['dem_fuzzymatch', 'dem_poll_avg', 'dem_effn']] = data.apply(lambda x: get_poll_avg(x['year'], x['state'], x['special'],
                                                                                        'DEM', x['dem_cand']), axis=1)
data[['rep_fuzzymatch', 'rep_poll_avg', 'rep_effn']] = data.apply(lambda x: get_poll_avg(x['year'], x['state'], x['special'],
                                                                                        'REP', x['rep_cand']), axis=1)
for party in ['dem', 'rep']:
    data[f'{party}_poll_avg'] = data[f'{party}_poll_avg'].fillna(0)
    data[f'{party}_effn'] = data[f'{party}_effn'].fillna(0)
data.head()

,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_fuzzymatch,dem_poll_avg,dem_effn,rep_fuzzymatch,rep_poll_avg,rep_effn
0,2014,Alabama,AL,False,NaN,795606.0,818090.0,[],Jeff Sessions,[],True,[],381899.0,NaN,NaN,NaN,0.0,381899.00,381899.00,0.000000,100.000000,NaN,0.000000,0.000000,NaN,0.000000,0.000000
1,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00,11986522.00,50.224502,49.775498,Mark Begich,44.935905,3.199893,Dan Sullivan,44.639290,3.199893
2,2014,Arkansas,AR,False,334174.0,478819.0,847505.0,Mark L. Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11,18893108.11,42.346680,57.653320,Mark L. Pryor,41.723965,3.170259,Tom Cotton,47.938130,3.170259
3,2014,Colorado,CO,False,944203.0,983891.0,2041058.0,Mark Udall,Cory Gardner,True,False,13414189.0,8725449.0,1928094.0,48.970797,51.029203,13414189.0,8725449.00,22139638.00,60.589017,39.410983,Mark Udall,44.062920,9.133357,Cory Gardner,45.824609,9.133357
4,2014,Delaware,DE,False,130655.0,98823.0,234038.0,Christopher A. Coons,Kevin Wade,True,False,2845059.0,92155.0,229478.0,56.935741,43.064259,2845059.0,92155.00,2937214.00,96.862503,3.137497,Christopher A. Coons,54.000000,0.469424,Kevin Wade,36.000000,0.469424


In [1178]:
data = data.drop(['dem_fuzzymatch', 'rep_fuzzymatch'], axis=1)

In [1179]:
urb = pd.read_csv('data/demo/urban_rural_pop/DECENNIALCD1182020.P2-Data-STATES.csv')
urb.columns = urb.iloc[0]
urb = urb.iloc[1:]
# urb = urb.rename({
#         'Geographic Area Name': 'state',
#         'Total!!Urban': 'urban',
#         'Total!!Rural': 'rural',
#         'Total': 'total'
#     }, axis=1)
urb.columns

Index([                           'Geography',
                       'Geographic Area Name',
                                  ' !!Total:',
                           ' !!Total:!!Urban',
                           ' !!Total:!!Rural',
       ' !!Total:!!Not defined for this file',
                                          nan],
      dtype='object', name=0)

In [1180]:
pres = pd.read_csv('data/pres_results/1976-2024-president.csv')
pres = pres[(pres['year'] >= 2008) & (~pres['candidate'].isin(['PAIJ BORING AND JAIMEY RUTSCHMAN', 'SETHATINA NEWMAN']))]
pres_pivot = pd.pivot_table(pres, values='candidatevotes', index=['year', 'state'], columns='party_simplified',
                           aggfunc='first')[['DEMOCRAT', 'REPUBLICAN']]
pres_pivot = pres_pivot.reset_index()
pres_pivot = pres_pivot[(~pres_pivot['DEMOCRAT'].isna()) | (~pres_pivot['REPUBLICAN'].isna())]
pres_pivot = pres_pivot.rename({
    'DEMOCRAT': 'dem',
    'REPUBLICAN': 'rep'
}, axis=1)
pres_pivot['tot'] = pres_pivot['dem'] + pres_pivot['rep']
for party in ['dem', 'rep']:
    pres_pivot[f'{party}_2p_pct'] = pres_pivot[party] / pres_pivot['tot'] * 100
pres_pivot['state_po'] = pres_pivot['state'].map(lambda x: x.title()).map(us_state_to_abbrev)
pres_pivot = pres_pivot[~pres_pivot['state_po'].isna()]
pres_pivot['year'] = pres_pivot['year'].astype(int)
pres_pivot['lean'] = pres_pivot['dem_2p_pct'] - pres_pivot['year'].astype(str).map(pop_vote)
pres_pivot.head(10)

party_simplified,year,state,dem,rep,tot,dem_2p_pct,rep_2p_pct,state_po,lean
0,2008,ALABAMA,813479.0,1266546.0,2080025.0,39.109097,60.890903,AL,-14.579750
1,2008,ALASKA,123594.0,193841.0,317435.0,38.935215,61.064785,AK,-14.753632
2,2008,ARIZONA,1034707.0,1230111.0,2264818.0,45.686099,54.313901,AZ,-8.002748
3,2008,ARKANSAS,422310.0,638017.0,1060327.0,39.828279,60.171721,AR,-13.860568
4,2008,CALIFORNIA,8274473.0,5011781.0,13286254.0,62.278450,37.721550,CA,8.589602
5,2008,COLORADO,1288576.0,1073589.0,2362165.0,54.550635,45.449365,CO,0.861787
6,2008,CONNECTICUT,997772.0,629428.0,1627200.0,61.318338,38.681662,CT,7.629491
7,2008,DELAWARE,255459.0,152374.0,407833.0,62.638139,37.361861,DE,8.949291
9,2008,FLORIDA,4282074.0,4045624.0,8327698.0,51.419660,48.580340,FL,-2.269187
10,2008,GEORGIA,1844123.0,2048759.0,3892882.0,47.371664,52.628336,GA,-6.317183


In [1181]:
joined_dfs = []

year_to_demo_df = {
    '2014': demo_14,
    '2016': demo_16,
    '2018': demo_18,
    '2020': demo_20,
    '2022': demo_22,
    '2024': demo_24
}

year_to_educinc_df = {
    '2014': educinc_2014,
    '2016': educinc_2016,
    '2018': educinc_2018,
    '2020': educinc_2020,
    '2022': educinc_2022,
    '2024': educinc_2024
}

year_to_urbrur_cong = {
    '2014': '116',
    '2016': '116',
    '2018': '116',
    '2020': '116',
    '2022': '118',
    '2024': '118'
}

for yr in np.unique(data['year']):
    #pvi = pd.read_csv(f'transformed/pvi/past_pres_results_by{yr % 2000}dist.csv')
    if yr in [2014, 2018, 2022, 2026]:
        prev_cyc = (yr % 2000) - ((yr % 2000) % 4)
    elif yr in [2016, 2020, 2024]:
        prev_cyc = (yr % 2000) - 4
    else:
        raise ValueError('Invalid year')
    prev_2cyc = prev_cyc - 4
    if yr in [2014, 2018, 2022, 2026]:
        most_rec_cyc = (yr % 2000) - 2
    elif yr in [2016, 2020, 2024]:
        most_rec_cyc = yr % 2000 ## THIS IS FINE - WE SUBTRACT 2 TO GET PREVIOUS CYCLE WHEN CALCULATING POLARIZATION FOR A SPECIFIC YEAR.
        ## THIS IS JUST TO HAVE COLUMNS ALIGN SO IT'S EASY TO CALCULATE CORRELATION
        ## DO NOT CHANGE THINKING THIS IS A MISTAKE
    else:
        raise ValueError('Invalid year')

    prev_cyc = int('20' + str(prev_cyc))
    prev_2cyc = int('20' + f"{'0' if prev_2cyc < 10 else ''}{prev_2cyc}")
    most_rec_cyc = int('20' + str(most_rec_cyc))
    #print(prev_cyc, prev_2cyc, most_rec_cyc)
    df = data[data['year'] == yr]
    df = pd.merge(left=df, right=pres_pivot[pres_pivot['year'] == prev_cyc][['state_po', 'lean']].rename({'lean': 'prev_lean'}, axis=1), 
                  on='state_po', how='left')
    df = pd.merge(left=df, right=pres_pivot[pres_pivot['year'] == prev_2cyc][['state_po', 'lean']].rename({'lean': 'prev2_lean'}, axis=1), 
                  on='state_po', how='left')
    df = pd.merge(left=df, right=pres_pivot[pres_pivot['year'] == most_rec_cyc][['state_po', 'dem_2p_pct']].rename({'dem_2p_pct': 'dem_2p_prev'}, axis=1),
                 on='state_po', how='left')
    df = pd.merge(left=df, right=pres_pivot[pres_pivot['year'] == most_rec_cyc-4][['state_po', 'dem_2p_pct']].rename({'dem_2p_pct': 'dem_2p_prev2'}, axis=1),
                 on='state_po', how='left')

    # Best to use interaction terms for presidential approval and ICS - otherwise we won't have consistent effect
    # For instance high ICS is good for Dems if Dem president is incumbent but bad for Dems if Republican president is incumbent
    #df['pres_approval_avg_x_incpres'] = df['pres_approval_avg'] * df['incumbent_pres']
    #df['ics_x_incpres'] = df['ics'] * df['incumbent_pres']

    # Racial demographics
    demo = year_to_demo_df[f'{yr}'].copy()
    demo['state_po'] = demo['state'].map(us_state_to_abbrev)
    demo = demo[['state_po', 'cvap_white_pct', 'cvap_aapi_pct', 'cvap_hisp_pct', 'cvap_black_pct', 'cvap_natam_pct']]
    #demo['cvap_pop'] = demo['cvap_pop'].astype(str).str.replace(',', '').astype(int)
    df = pd.merge(left=df, right=demo, left_on='state_po', right_on='state_po', how='left')

    # Educational demographics
    educ = year_to_educinc_df[f'{yr}'].copy()
    educ['state_po'] = educ['State'].map(us_state_to_abbrev)
    educ = educ[['state_po', 'college']]
    df = pd.merge(left=df, right=educ, left_on='state_po', right_on='state_po', how='left')

    # Urban/rural pop
    census_yr = 2010 if yr <= 2020 else 2020
    cong = year_to_urbrur_cong[f'{yr}']
    urb = pd.read_csv(f'data/demo/urban_rural_pop/DECENNIALCD{cong}{census_yr}.P2-Data-STATES.csv')
    urb.columns = urb.iloc[0]
    urb = urb.iloc[1:]
    if census_yr == 2010:
        if cong == '116':
            urb = urb[['Label for GEO_ID', 'Total!!Urban', 'Total!!Rural', 'Total']]
            urb = urb.rename({'Label for GEO_ID': 'Geographic Area Name'}, axis=1)
        else:
            urb = urb[['Geographic Area Name', 'Total!!Urban', 'Total!!Rural', 'Total']]
        urb = urb.rename({
            'Geographic Area Name': 'state',
            'Total!!Urban': 'urban',
            'Total!!Rural': 'rural',
            'Total': 'total'
        }, axis=1)
    elif census_yr == 2020:
        urb.columns = urb.columns.str.strip()
        urb = urb[['Geographic Area Name', '!!Total:!!Urban', '!!Total:!!Rural', '!!Total:']]
        urb = urb.rename({
            'Geographic Area Name': 'state',
            '!!Total:!!Urban': 'urban',
            '!!Total:!!Rural': 'rural',
            '!!Total:': 'total'
        }, axis=1)
    else:
        raise ValueError('Invalid year.')
    urb['total'] = urb['urban'].astype(int) + urb['rural'].astype(int)
    #print(yr, urb.columns)
    urb['urban'] = urb['urban'].astype(int)
    urb['rural'] = urb['rural'].astype(int)
    urb['total'] = urb['total'].astype(int)
    urb['urban_pct'] = urb['urban'] / urb['total'] * 100
    urb['rural_pct'] = urb['rural'] / urb['total'] * 100
    urb['state_po'] = urb['state'].map(us_state_to_abbrev)
    df = pd.merge(left=df, right=urb[['state_po', 'urban', 'rural', 'urban_pct', 'rural_pct']], on='state_po', how='left')
    

    ## Geographic area (for population density calculation)
    # geo = shapefile_gdfs[f'{yr}'].copy() # Yes it's CRS units in area, but considering we're utilizing pop density for k-means clustering,
    ## what matters most is the *relative* densities are correct
    # geo['Code'] = geo['Code'].astype(str).map(lambda x: x[:2] + '-00' if x[3:] == 'AL' else x)
    # df = pd.merge(left=df, right=geo[['Code', 'area']], left_on='district', right_on='Code', how='left')

    # Get 2024 district demographics (useful for prediction)
    if yr == 2024:
        demog = pd.merge(left=demo, right=educ, left_on='state_po', right_on='state_po', how='left')
        demog = pd.merge(left=demog, right=urb, left_on='state_po', right_on='state_po', how='left')
        demog.to_csv('2026_data/senate_demographics.csv')
            
    joined_dfs.append(df)

mdata = pd.concat(joined_dfs, axis=0)
mdata.head()

,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,prev_lean,prev2_lean,dem_2p_prev,dem_2p_prev2,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,urban_pct,rural_pct
0,2014,Alabama,AL,False,NaN,795606.0,818090.0,[],Jeff Sessions,[],True,[],381899.0,NaN,NaN,NaN,0.0,381899.00,381899.00,0.000000,100.000000,0.000000,0.000000,0.000000,0.000000,-13.180091,-14.579750,38.783771,39.109097,70.428914,0.724167,1.623772,25.782783,0.523703,23.1,2821804,1957932,59.036817,40.963183
1,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00,11986522.00,50.224502,49.775498,44.935905,3.199893,44.639290,3.199893,-9.279153,-14.753632,42.684710,38.935215,68.78786,4.803705,4.887711,3.344598,13.465866,27.7,468893,241338,66.019788,33.980212
2,2014,Arkansas,AR,False,334174.0,478819.0,847505.0,Mark L. Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11,18893108.11,42.346680,57.653320,41.723965,3.170259,47.938130,3.170259,-14.118268,-13.860568,37.845595,39.828279,79.509688,0.804425,2.788077,15.007081,0.624668,20.6,1637589,1278329,56.160324,43.839676
3,2014,Colorado,CO,False,944203.0,983891.0,2041058.0,Mark Udall,Cory Gardner,True,False,13414189.0,8725449.0,1928094.0,48.970797,51.029203,13414189.0,8725449.00,22139638.00,60.589017,39.410983,44.062920,9.133357,45.824609,9.133357,0.784126,0.861787,52.747988,54.550635,77.59865,2.210174,14.093498,3.831403,0.962621,37.5,4332761,696435,86.152160,13.847840
4,2014,Delaware,DE,False,130655.0,98823.0,234038.0,Christopher A. Coons,Kevin Wade,True,False,2845059.0,92155.0,229478.0,56.935741,43.064259,2845059.0,92155.00,2937214.00,96.862503,3.137497,54.000000,0.469424,36.000000,0.469424,7.483093,8.949291,59.446955,62.638139,71.087114,2.115118,4.805171,20.679898,0.360503,29.4,747949,149985,83.296657,16.703343


In [1182]:
mdata['geography'] = mdata['state'] + mdata['special'].map(lambda x: ' Special' if x == True else '')

In [1183]:
clust_model = joblib.load('model/demographic_kmeans.pkl')
mdata['demo_cluster'] = clust_model.predict(mdata[['cvap_white_pct', 'cvap_aapi_pct', 'cvap_black_pct', 'cvap_hisp_pct', 'cvap_natam_pct', 'college',
                                                        'urban_pct']])

In [1184]:
for party in ['dem', 'rep']:
    mdata[f'{party}_funds_2p_pct'] = mdata[f'{party}_funds_2p_pct'].fillna(0)

In [1185]:
# Exclude uncontested races and races w/ all candidates from one party
mdata['no_dem_cand_flag'] = mdata['dem_cand'].map(lambda x: x == '[]')
mdata['no_rep_cand_flag'] = mdata['rep_cand'].map(lambda x: x == '[]')
uncontested = mdata[(mdata['no_dem_cand_flag']) | (mdata['no_rep_cand_flag'])]
mdata = mdata[~mdata['no_dem_cand_flag']]
mdata = mdata[~mdata['no_rep_cand_flag']]
mdata.head()

,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,prev_lean,prev2_lean,dem_2p_prev,dem_2p_prev2,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,urban_pct,rural_pct,geography,demo_cluster,no_dem_cand_flag,no_rep_cand_flag
1,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00,11986522.00,50.224502,49.775498,44.935905,3.199893,44.639290,3.199893,-9.279153,-14.753632,42.684710,38.935215,68.78786,4.803705,4.887711,3.344598,13.465866,27.7,468893,241338,66.019788,33.980212,Alaska,5,False,False
2,2014,Arkansas,AR,False,334174.0,478819.0,847505.0,Mark L. Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11,18893108.11,42.346680,57.653320,41.723965,3.170259,47.938130,3.170259,-14.118268,-13.860568,37.845595,39.828279,79.509688,0.804425,2.788077,15.007081,0.624668,20.6,1637589,1278329,56.160324,43.839676,Arkansas,5,False,False
3,2014,Colorado,CO,False,944203.0,983891.0,2041058.0,Mark Udall,Cory Gardner,True,False,13414189.0,8725449.0,1928094.0,48.970797,51.029203,13414189.0,8725449.00,22139638.00,60.589017,39.410983,44.062920,9.133357,45.824609,9.133357,0.784126,0.861787,52.747988,54.550635,77.59865,2.210174,14.093498,3.831403,0.962621,37.5,4332761,696435,86.152160,13.847840,Colorado,2,False,False
4,2014,Delaware,DE,False,130655.0,98823.0,234038.0,Christopher A. Coons,Kevin Wade,True,False,2845059.0,92155.0,229478.0,56.935741,43.064259,2845059.0,92155.00,2937214.00,96.862503,3.137497,54.000000,0.469424,36.000000,0.469424,7.483093,8.949291,59.446955,62.638139,71.087114,2.115118,4.805171,20.679898,0.360503,29.4,747949,149985,83.296657,16.703343,Delaware,2,False,False
5,2014,Georgia,GA,False,1160811.0,1358088.0,2567805.0,M. Michelle Nunn,David A. Perdue,False,False,13657065.0,8248614.0,2518899.0,46.084063,53.915937,13657065.0,8248614.00,21905679.00,62.344860,37.655140,45.051104,8.728368,47.118414,8.728368,-5.920511,-6.317183,46.043351,47.371664,61.646427,2.310356,3.83251,31.101709,0.231023,28.3,7272151,2415502,75.066180,24.933820,Georgia,5,False,False


In [1186]:
for col in ['dem_inc', 'rep_inc']:
    mdata[col] = mdata[col].map(lambda x: ast.literal_eval(x))
mdata['dem_inc_any'] = mdata['dem_inc'].map(lambda x: x if not isinstance(x, list) else (True if True in x else False))
mdata['rep_inc_any'] = mdata['rep_inc'].map(lambda x: x if not isinstance(x, list) else (True if True in x else False))

In [1187]:
mdata['dem_inc_dummy'] = mdata['dem_inc_any'].map(lambda x: 1 if x == True else 0)
mdata['rep_inc_dummy'] = mdata['rep_inc_any'].map(lambda x: 1 if x == True else 0)

In [1188]:
mdata['pvi'] = mdata['prev_lean'] * 0.75 + mdata['prev2_lean'] * 0.25
mdata.head()

,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,prev_lean,prev2_lean,dem_2p_prev,dem_2p_prev2,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,urban_pct,rural_pct,geography,demo_cluster,no_dem_cand_flag,no_rep_cand_flag,dem_inc_any,rep_inc_any,dem_inc_dummy,rep_inc_dummy,pvi
1,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00,11986522.00,50.224502,49.775498,44.935905,3.199893,44.639290,3.199893,-9.279153,-14.753632,42.684710,38.935215,68.78786,4.803705,4.887711,3.344598,13.465866,27.7,468893,241338,66.019788,33.980212,Alaska,5,False,False,True,False,1,0,-10.647773
2,2014,Arkansas,AR,False,334174.0,478819.0,847505.0,Mark L. Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11,18893108.11,42.346680,57.653320,41.723965,3.170259,47.938130,3.170259,-14.118268,-13.860568,37.845595,39.828279,79.509688,0.804425,2.788077,15.007081,0.624668,20.6,1637589,1278329,56.160324,43.839676,Arkansas,5,False,False,True,False,1,0,-14.053843
3,2014,Colorado,CO,False,944203.0,983891.0,2041058.0,Mark Udall,Cory Gardner,True,False,13414189.0,8725449.0,1928094.0,48.970797,51.029203,13414189.0,8725449.00,22139638.00,60.589017,39.410983,44.062920,9.133357,45.824609,9.133357,0.784126,0.861787,52.747988,54.550635,77.59865,2.210174,14.093498,3.831403,0.962621,37.5,4332761,696435,86.152160,13.847840,Colorado,2,False,False,True,False,1,0,0.803541
4,2014,Delaware,DE,False,130655.0,98823.0,234038.0,Christopher A. Coons,Kevin Wade,True,False,2845059.0,92155.0,229478.0,56.935741,43.064259,2845059.0,92155.00,2937214.00,96.862503,3.137497,54.000000,0.469424,36.000000,0.469424,7.483093,8.949291,59.446955,62.638139,71.087114,2.115118,4.805171,20.679898,0.360503,29.4,747949,149985,83.296657,16.703343,Delaware,2,False,False,True,False,1,0,7.849642
5,2014,Georgia,GA,False,1160811.0,1358088.0,2567805.0,M. Michelle Nunn,David A. Perdue,False,False,13657065.0,8248614.0,2518899.0,46.084063,53.915937,13657065.0,8248614.00,21905679.00,62.344860,37.655140,45.051104,8.728368,47.118414,8.728368,-5.920511,-6.317183,46.043351,47.371664,61.646427,2.310356,3.83251,31.101709,0.231023,28.3,7272151,2415502,75.066180,24.933820,Georgia,5,False,False,False,False,0,0,-6.019679


In [1189]:
state_to_census_region = {
    'Connecticut': 'New England',
    'Maine': 'New England',
    'Massachusetts': 'New England',
    'New Hampshire': 'New England',
    'Rhode Island': 'New England',
    'Vermont': 'New England',
    'New Jersey': 'Mid-Atlantic',
    'New York': 'Mid-Atlantic',
    'Pennsylvania': 'Mid-Atlantic',
    'Illinois': 'East North Central',
    'Indiana': 'East North Central',
    'Michigan': 'East North Central',
    'Ohio': 'East North Central',
    'Wisconsin': 'East North Central',
    'Iowa': 'West North Central',
    'Kansas': 'West North Central',
    'Minnesota': 'West North Central',
    'Missouri': 'West North Central',
    'Nebraska': 'West North Central',
    'North Dakota': 'West North Central',
    'South Dakota': 'West North Central',
    'Delaware': 'South Atlantic',
    'Maryland': 'South Atlantic',
    'Florida': 'South Atlantic',
    'Georgia': 'South Atlantic',
    'North Carolina': 'South Atlantic',
    'South Carolina': 'South Atlantic',
    'Virginia': 'South Atlantic',
    'West Virginia': 'South Atlantic',
    'Alabama': 'East South Central',
    'Kentucky': 'East South Central',
    'Mississippi': 'East South Central',
    'Tennessee': 'East South Central',
    'Arkansas': 'West South Central',
    'Louisiana': 'West South Central',
    'Oklahoma': 'West South Central',
    'Texas': 'West South Central',
    'Arizona': 'Mountain',
    'Colorado': 'Mountain',
    'Idaho': 'Mountain',
    'Montana': 'Mountain',
    'Nevada': 'Mountain',
    'New Mexico': 'Mountain',
    'Utah': 'Mountain',
    'Wyoming': 'Mountain',
    'California': 'Pacific',
    'Alaska': 'Pacific',
    'Oregon': 'Pacific',
    'Washington': 'Pacific',
    'Hawaii': 'Pacific'
}

In [1190]:
mdata['census_region'] = mdata['state'].map(state_to_census_region)

In [1191]:
## Political scandals
# List of scandals by federal and state officeholders courtesy of Nathaniel Rakich
scandals = pd.read_csv('https://docs.google.com/spreadsheets/d/1ksBLxRR3GCZd33IvhkcNqqBd5K8HwlWC7YuAkVmS1lg/export?format=csv')
scandals.to_csv('data/rakich_scandals.csv')
scandals = scandals.set_axis(['politician', 'state', 'position', 'year', 'type', 'outcome', 'comeback'], axis=1)
scandals['politician'] = scandals['politician'].map(lambda x: unicodedata.normalize('NFC', x))
scandals.head()

,politician,state,position,year,type,outcome,comeback
0,Dale Caldwell,New Jersey,Lieutenant Governor,2026,"Sexual harassment, ethics",TBD,NaN
1,Jimmy Gomez,California,Representative,2026,Affair with staffer,TBD,NaN
2,Liz Murrill,Louisiana,Attorney General,2026,Intimidation,TBD,NaN
3,Jim Costa,California,Representative,2026,Sexual harassment,TBD,NaN
4,Chuck Edwards,North Carolina,Representative,2026,Sexual harassment,TBD,NaN


In [1192]:
def get_scandals(politician, state, cycle):
    # :param cycle: The year of the election cycle
    mask = ((scandals['state'] == state) & (scandals['year'] <= cycle))
    scan_df = scandals[mask]
    fuzzymatch = process.extractOne(politician, scan_df['politician'], scorer=fuzz.WRatio, score_cutoff=90)
    if fuzzymatch is None:
        return 0
    scan_df_pol = scan_df[scan_df['politician'] == fuzzymatch[0]]
    # Exponential decay factor to put less weight on older scandals relative to newer scandals
    # e^(-\delta_t/4)
    scan_df_pol['decay_factor'] = np.exp(-(scan_df_pol['year'].astype(int).map(lambda x: cycle - x))/4)
    pol_scandal_score = scan_df_pol.groupby(['politician']).agg({'decay_factor': 'sum'})['decay_factor'].values[0]
    return pol_scandal_score

In [1193]:
mdata['dem_scandal_score'] = mdata.apply(lambda x: get_scandals(x['dem_cand'], x['state'], x['year']), axis=1)
mdata['rep_scandal_score'] = mdata.apply(lambda x: get_scandals(x['rep_cand'], x['state'], x['year']), axis=1)
mdata['net_scandal_score'] = mdata['dem_scandal_score'] - mdata['rep_scandal_score']
mdata.head()

,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,prev_lean,prev2_lean,dem_2p_prev,dem_2p_prev2,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,urban_pct,rural_pct,geography,demo_cluster,no_dem_cand_flag,no_rep_cand_flag,dem_inc_any,rep_inc_any,dem_inc_dummy,rep_inc_dummy,pvi,census_region,dem_scandal_score,rep_scandal_score,net_scandal_score
1,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00,11986522.00,50.224502,49.775498,44.935905,3.199893,44.639290,3.199893,-9.279153,-14.753632,42.684710,38.935215,68.78786,4.803705,4.887711,3.344598,13.465866,27.7,468893,241338,66.019788,33.980212,Alaska,5,False,False,True,False,1,0,-10.647773,Pacific,0.0,0.0,0.0
2,2014,Arkansas,AR,False,334174.0,478819.0,847505.0,Mark L. Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11,18893108.11,42.346680,57.653320,41.723965,3.170259,47.938130,3.170259,-14.118268,-13.860568,37.845595,39.828279,79.509688,0.804425,2.788077,15.007081,0.624668,20.6,1637589,1278329,56.160324,43.839676,Arkansas,5,False,False,True,False,1,0,-14.053843,West South Central,0.0,0.0,0.0
3,2014,Colorado,CO,False,944203.0,983891.0,2041058.0,Mark Udall,Cory Gardner,True,False,13414189.0,8725449.0,1928094.0,48.970797,51.029203,13414189.0,8725449.00,22139638.00,60.589017,39.410983,44.062920,9.133357,45.824609,9.133357,0.784126,0.861787,52.747988,54.550635,77.59865,2.210174,14.093498,3.831403,0.962621,37.5,4332761,696435,86.152160,13.847840,Colorado,2,False,False,True,False,1,0,0.803541,Mountain,0.0,0.0,0.0
4,2014,Delaware,DE,False,130655.0,98823.0,234038.0,Christopher A. Coons,Kevin Wade,True,False,2845059.0,92155.0,229478.0,56.935741,43.064259,2845059.0,92155.00,2937214.00,96.862503,3.137497,54.000000,0.469424,36.000000,0.469424,7.483093,8.949291,59.446955,62.638139,71.087114,2.115118,4.805171,20.679898,0.360503,29.4,747949,149985,83.296657,16.703343,Delaware,2,False,False,True,False,1,0,7.849642,South Atlantic,0.0,0.0,0.0
5,2014,Georgia,GA,False,1160811.0,1358088.0,2567805.0,M. Michelle Nunn,David A. Perdue,False,False,13657065.0,8248614.0,2518899.0,46.084063,53.915937,13657065.0,8248614.00,21905679.00,62.344860,37.655140,45.051104,8.728368,47.118414,8.728368,-5.920511,-6.317183,46.043351,47.371664,61.646427,2.310356,3.83251,31.101709,0.231023,28.3,7272151,2415502,75.066180,24.933820,Georgia,5,False,False,False,False,0,0,-6.019679,South Atlantic,0.0,0.0,0.0


In [1194]:
stwar = pd.read_csv('data/split_ticket_war_dataset.csv')
stwar = stwar[stwar['Chamber'] == 'Senate']
stwar['Year']  = stwar['Year'].astype(int)
stwar.head()

,Year,Chamber,Geography,Democrat,Republican,WAR,Sortable
398,2024,Senate,Arizona,Ruben Gallego,Kari Lake,D+7.2,-7.2
399,2024,Senate,California,Adam Schiff,Steve Garvey,R+4.1,4.1
400,2024,Senate,Connecticut,Chris Murphy,Matthew Corey,R+0.1,0.1
401,2024,Senate,Delaware,Lisa Blunt Rochester,Eric Hansen,D+1.4,-1.4
402,2024,Senate,Florida,Debbie Mucarsel-Powell,Rick Scott,D+1.6,-1.6


In [1195]:
mdata = pd.merge(left=mdata, right=stwar[['Year', 'Geography', 'Democrat', 'Republican']], left_on=['year', 'geography'], right_on=['Year', 'Geography'],
               how='left')
mdata.head()

,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,prev_lean,prev2_lean,dem_2p_prev,dem_2p_prev2,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,urban_pct,rural_pct,geography,demo_cluster,no_dem_cand_flag,no_rep_cand_flag,dem_inc_any,rep_inc_any,dem_inc_dummy,rep_inc_dummy,pvi,census_region,dem_scandal_score,rep_scandal_score,net_scandal_score,Year,Geography,Democrat,Republican
0,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00,11986522.00,50.224502,49.775498,44.935905,3.199893,44.639290,3.199893,-9.279153,-14.753632,42.684710,38.935215,68.78786,4.803705,4.887711,3.344598,13.465866,27.7,468893,241338,66.019788,33.980212,Alaska,5,False,False,True,False,1,0,-10.647773,Pacific,0.0,0.0,0.0,2014.0,Alaska,Mark Begich,Dan Sullivan
1,2014,Arkansas,AR,False,334174.0,478819.0,847505.0,Mark L. Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11,18893108.11,42.346680,57.653320,41.723965,3.170259,47.938130,3.170259,-14.118268,-13.860568,37.845595,39.828279,79.509688,0.804425,2.788077,15.007081,0.624668,20.6,1637589,1278329,56.160324,43.839676,Arkansas,5,False,False,True,False,1,0,-14.053843,West South Central,0.0,0.0,0.0,2014.0,Arkansas,Mark Pryor,Tom Cotton
2,2014,Colorado,CO,False,944203.0,983891.0,2041058.0,Mark Udall,Cory Gardner,True,False,13414189.0,8725449.0,1928094.0,48.970797,51.029203,13414189.0,8725449.00,22139638.00,60.589017,39.410983,44.062920,9.133357,45.824609,9.133357,0.784126,0.861787,52.747988,54.550635,77.59865,2.210174,14.093498,3.831403,0.962621,37.5,4332761,696435,86.152160,13.847840,Colorado,2,False,False,True,False,1,0,0.803541,Mountain,0.0,0.0,0.0,2014.0,Colorado,Mark Udall,Cory Gardner
3,2014,Delaware,DE,False,130655.0,98823.0,234038.0,Christopher A. Coons,Kevin Wade,True,False,2845059.0,92155.0,229478.0,56.935741,43.064259,2845059.0,92155.00,2937214.00,96.862503,3.137497,54.000000,0.469424,36.000000,0.469424,7.483093,8.949291,59.446955,62.638139,71.087114,2.115118,4.805171,20.679898,0.360503,29.4,747949,149985,83.296657,16.703343,Delaware,2,False,False,True,False,1,0,7.849642,South Atlantic,0.0,0.0,0.0,2014.0,Delaware,Chris Coons,Kevin L. Wade
4,2014,Georgia,GA,False,1160811.0,1358088.0,2567805.0,M. Michelle Nunn,David A. Perdue,False,False,13657065.0,8248614.0,2518899.0,46.084063,53.915937,13657065.0,8248614.00,21905679.00,62.344860,37.655140,45.051104,8.728368,47.118414,8.728368,-5.920511,-6.317183,46.043351,47.371664,61.646427,2.310356,3.83251,31.101709,0.231023,28.3,7272151,2415502,75.066180,24.933820,Georgia,5,False,False,False,False,0,0,-6.019679,South Atlantic,0.0,0.0,0.0,2014.0,Georgia,Michelle Nunn,David Perdue


In [1196]:
mdata = mdata.rename({'Democrat': 'dem_stand', 'Republican': 'rep_stand'}, axis=1)
for party in ['dem', 'rep']:
    mdata[f'{party}_stand'] = mdata[f'{party}_stand'].fillna(mdata[f'{party}_cand'])
    mdata[f'{party}_cand'] = mdata[f'{party}_stand']

In [1197]:
mdata['inc_dummy'] = mdata['dem_inc_dummy'] - mdata['rep_inc_dummy']
mdata.head()

,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,prev_lean,prev2_lean,dem_2p_prev,dem_2p_prev2,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,urban_pct,rural_pct,geography,demo_cluster,no_dem_cand_flag,no_rep_cand_flag,dem_inc_any,rep_inc_any,dem_inc_dummy,rep_inc_dummy,pvi,census_region,dem_scandal_score,rep_scandal_score,net_scandal_score,Year,Geography,dem_stand,rep_stand,inc_dummy
0,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00,11986522.00,50.224502,49.775498,44.935905,3.199893,44.639290,3.199893,-9.279153,-14.753632,42.684710,38.935215,68.78786,4.803705,4.887711,3.344598,13.465866,27.7,468893,241338,66.019788,33.980212,Alaska,5,False,False,True,False,1,0,-10.647773,Pacific,0.0,0.0,0.0,2014.0,Alaska,Mark Begich,Dan Sullivan,1
1,2014,Arkansas,AR,False,334174.0,478819.0,847505.0,Mark Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11,18893108.11,42.346680,57.653320,41.723965,3.170259,47.938130,3.170259,-14.118268,-13.860568,37.845595,39.828279,79.509688,0.804425,2.788077,15.007081,0.624668,20.6,1637589,1278329,56.160324,43.839676,Arkansas,5,False,False,True,False,1,0,-14.053843,West South Central,0.0,0.0,0.0,2014.0,Arkansas,Mark Pryor,Tom Cotton,1
2,2014,Colorado,CO,False,944203.0,983891.0,2041058.0,Mark Udall,Cory Gardner,True,False,13414189.0,8725449.0,1928094.0,48.970797,51.029203,13414189.0,8725449.00,22139638.00,60.589017,39.410983,44.062920,9.133357,45.824609,9.133357,0.784126,0.861787,52.747988,54.550635,77.59865,2.210174,14.093498,3.831403,0.962621,37.5,4332761,696435,86.152160,13.847840,Colorado,2,False,False,True,False,1,0,0.803541,Mountain,0.0,0.0,0.0,2014.0,Colorado,Mark Udall,Cory Gardner,1
3,2014,Delaware,DE,False,130655.0,98823.0,234038.0,Chris Coons,Kevin L. Wade,True,False,2845059.0,92155.0,229478.0,56.935741,43.064259,2845059.0,92155.00,2937214.00,96.862503,3.137497,54.000000,0.469424,36.000000,0.469424,7.483093,8.949291,59.446955,62.638139,71.087114,2.115118,4.805171,20.679898,0.360503,29.4,747949,149985,83.296657,16.703343,Delaware,2,False,False,True,False,1,0,7.849642,South Atlantic,0.0,0.0,0.0,2014.0,Delaware,Chris Coons,Kevin L. Wade,1
4,2014,Georgia,GA,False,1160811.0,1358088.0,2567805.0,Michelle Nunn,David Perdue,False,False,13657065.0,8248614.0,2518899.0,46.084063,53.915937,13657065.0,8248614.00,21905679.00,62.344860,37.655140,45.051104,8.728368,47.118414,8.728368,-5.920511,-6.317183,46.043351,47.371664,61.646427,2.310356,3.83251,31.101709,0.231023,28.3,7272151,2415502,75.066180,24.933820,Georgia,5,False,False,False,False,0,0,-6.019679,South Atlantic,0.0,0.0,0.0,2014.0,Georgia,Michelle Nunn,David Perdue,0


In [1198]:
house = pd.read_csv('transformed/all_2p_house_races_trainset.csv')
house.head()

,Unnamed: 0,year,state,state_po,special,district,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,seat_number,prev_lean,prev2_lean,dem_2p_prev,dem_2p_prev2,generic_ballot_avg,ics,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,urban_pct,rural_pct,dem_2p_12,demo_cluster,district_id,no_dem_cand_flag,no_rep_cand_flag,dem_inc_any,rep_inc_any,dem_inc_dummy,rep_inc_dummy,pvi,census_region,dem_scandal_score,rep_scandal_score,net_scandal_score,Year,Geography,dem_stand,rep_stand,inc_dummy,polarization
0,0,2014,Alabama,AL,False,AL-01,48278.0,103758.0,152234,Burton LeFlore,Bradley Byrne,False,True,33474.0,914507.3,152036.0,31.754321,68.245679,33474.00,914507.30,947981.30,3.531082,96.468918,0.0,0.0,0.0,0.0,1,-14.281877,-14.995432,37.681985,38.693415,2.645942,88.8,69.086637,0.821617,1.699690,26.504568,0.958585,22.4,456228.0,226592.0,66.815266,33.184734,NaN,5,AL01,False,False,False,True,0,1,-14.460266,East South Central,0.0,0.0,0.0,2014.0,AL-01,Burton LeFlore,Bradley Byrne,-1,0.875383
1,1,2014,Alabama,AL,False,AL-02,54692.0,113103.0,167952,Erick Wright,Martha Roby,False,True,6674.0,454019.64,167795.0,32.594535,67.405465,6674.00,454019.64,460693.64,1.448685,98.551315,0.0,0.0,0.0,0.0,2,-15.288962,-18.500155,36.674900,35.188692,2.645942,88.8,66.811928,0.736959,1.721323,29.271262,0.445915,21.3,373585.0,309235.0,54.712076,45.287924,NaN,5,AL02,False,False,False,True,0,1,-16.091761,East South Central,0.0,0.0,0.0,2014.0,AL-02,Erick Wright,Martha Roby,-1,0.875383
2,2,2014,Alabama,AL,False,AL-03,52816.0,103558.0,156620,Jesse Smith,Mike Rogers,False,True,2100.0,504511.67,156374.0,33.775436,66.224564,2100.00,504511.67,506611.67,0.414519,99.585481,0.0,0.0,0.0,0.0,3,-14.821901,-16.778726,37.141962,36.910121,2.645942,88.8,71.985754,0.582895,1.511632,24.982584,0.309579,20.1,343078.0,339741.0,50.244355,49.755645,NaN,5,AL03,False,False,False,True,0,1,-15.311107,East South Central,0.0,0.0,0.0,2014.0,AL-03,Jesse Smith,Mike Rogers,-1,0.875383
3,3,2014,Alabama,AL,False,AL-06,42291.0,135945.0,178449,Mark Lester,Gary Palmer,False,False,143501.09,1618984.26,178236.0,23.727530,76.272470,143501.09,1618984.26,1762485.35,8.141973,91.858027,0.0,0.0,0.0,0.0,6,-27.045432,-28.445196,24.918430,25.243651,2.645942,88.8,82.659657,1.063261,1.453968,13.741643,0.322422,34.1,472327.0,210492.0,69.173090,30.826910,NaN,2,AL06,False,False,False,False,0,0,-27.395373,East South Central,0.0,0.0,0.0,2014.0,AL-06,Mark Lester,Gary Palmer,0,0.875383
4,4,2014,Alaska,AK,False,AK-00,114602.0,142572.0,279741,Forrest Dunbar,Don Young,False,True,233131.98,461857.82,257174.0,44.562047,55.437953,233131.98,461857.82,694989.80,33.544662,66.455338,0.0,0.0,0.0,0.0,0,-9.279153,-14.753632,42.684710,38.935215,2.645942,88.8,68.787860,4.803705,4.887711,3.344598,13.465866,27.7,468893.0,241338.0,66.019788,33.980212,NaN,5,AK00,False,False,False,True,0,1,-10.647773,Pacific,0.0,1.0,-1.0,2014.0,AK-00,Forrest Dunbar,Don Young,-1,0.875383


In [1199]:
mdata = pd.merge(left=mdata, right=house[['year', 'generic_ballot_avg', 'polarization']].value_counts().reset_index(), on='year', how='left')
mdata.head(3)

,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,prev_lean,prev2_lean,dem_2p_prev,dem_2p_prev2,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,urban_pct,rural_pct,geography,demo_cluster,no_dem_cand_flag,no_rep_cand_flag,dem_inc_any,rep_inc_any,dem_inc_dummy,rep_inc_dummy,pvi,census_region,dem_scandal_score,rep_scandal_score,net_scandal_score,Year,Geography,dem_stand,rep_stand,inc_dummy,generic_ballot_avg,polarization,count
0,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00,11986522.00,50.224502,49.775498,44.935905,3.199893,44.639290,3.199893,-9.279153,-14.753632,42.684710,38.935215,68.78786,4.803705,4.887711,3.344598,13.465866,27.7,468893,241338,66.019788,33.980212,Alaska,5,False,False,True,False,1,0,-10.647773,Pacific,0.0,0.0,0.0,2014.0,Alaska,Mark Begich,Dan Sullivan,1,2.645942,0.875383,358
1,2014,Arkansas,AR,False,334174.0,478819.0,847505.0,Mark Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11,18893108.11,42.346680,57.653320,41.723965,3.170259,47.938130,3.170259,-14.118268,-13.860568,37.845595,39.828279,79.509688,0.804425,2.788077,15.007081,0.624668,20.6,1637589,1278329,56.160324,43.839676,Arkansas,5,False,False,True,False,1,0,-14.053843,West South Central,0.0,0.0,0.0,2014.0,Arkansas,Mark Pryor,Tom Cotton,1,2.645942,0.875383,358
2,2014,Colorado,CO,False,944203.0,983891.0,2041058.0,Mark Udall,Cory Gardner,True,False,13414189.0,8725449.0,1928094.0,48.970797,51.029203,13414189.0,8725449.00,22139638.00,60.589017,39.410983,44.062920,9.133357,45.824609,9.133357,0.784126,0.861787,52.747988,54.550635,77.59865,2.210174,14.093498,3.831403,0.962621,37.5,4332761,696435,86.152160,13.847840,Colorado,2,False,False,True,False,1,0,0.803541,Mountain,0.0,0.0,0.0,2014.0,Colorado,Mark Udall,Cory Gardner,1,2.645942,0.875383,358


In [1200]:
house[['year', 'generic_ballot_avg', 'polarization']].value_counts().reset_index()

,year,generic_ballot_avg,polarization,count
0,2020,-7.047614,0.974298,408
1,2022,1.021178,0.986203,401
2,2024,-0.835244,0.982821,398
3,2018,-7.921785,0.946717,394
4,2016,-2.991987,0.941915,372
5,2014,2.645942,0.875383,358


In [1201]:
house = house.rename({
    'district': 'geography'
}, axis=1)
mdata['chamber'] = np.full(mdata.shape[0], 'Senate')
house['chamber'] = np.full(house.shape[0], 'House')
house['special'] = np.full(house.shape[0], False)
mdata.head(1)

,year,state,state_po,special,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,prev_lean,prev2_lean,dem_2p_prev,dem_2p_prev2,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,urban_pct,rural_pct,geography,demo_cluster,no_dem_cand_flag,no_rep_cand_flag,dem_inc_any,rep_inc_any,dem_inc_dummy,rep_inc_dummy,pvi,census_region,dem_scandal_score,rep_scandal_score,net_scandal_score,Year,Geography,dem_stand,rep_stand,inc_dummy,generic_ballot_avg,polarization,count,chamber
0,2014,Alaska,AK,False,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.0,11986522.0,50.224502,49.775498,44.935905,3.199893,44.63929,3.199893,-9.279153,-14.753632,42.68471,38.935215,68.78786,4.803705,4.887711,3.344598,13.465866,27.7,468893,241338,66.019788,33.980212,Alaska,5,False,False,True,False,1,0,-10.647773,Pacific,0.0,0.0,0.0,2014.0,Alaska,Mark Begich,Dan Sullivan,1,2.645942,0.875383,358,Senate


In [1202]:
order = ['year', 'state', 'state_po', 'special', 'geography', 'dem', 'rep', 'totalvotes',
                                         'dem_cand', 'rep_cand', 'dem_inc', 'rep_inc', 'dem_funds',
                                         'rep_funds', '2party_votes', 'dem_pct_2p', 'rep_pct_2p',
                                         'dem_tot_funds', 'rep_tot_funds', 'tot_funds', 'dem_funds_2p_pct',
                                         'rep_funds_2p_pct', 'dem_poll_avg', 'dem_effn', 'rep_poll_avg',
                                         'rep_effn', 'prev_lean', 'prev2_lean', 'dem_2p_prev', 'dem_2p_prev2',
                                         'cvap_white_pct', 'cvap_aapi_pct', 'cvap_hisp_pct', 'cvap_black_pct',
                                         'cvap_natam_pct', 'college', 'urban', 'rural', 'urban_pct', 'rural_pct',
                                         'demo_cluster', 'no_dem_cand_flag', 'no_rep_cand_flag', 'dem_inc_any',
                                         'rep_inc_any', 'dem_inc_dummy', 'rep_inc_dummy', 'pvi',
                                         'census_region', 'dem_scandal_score', 'rep_scandal_score',
                                         'net_scandal_score', 'dem_stand', 'rep_stand', 'inc_dummy', 'chamber', 'generic_ballot_avg',
        'polarization']
mdata = pd.concat([mdata[order], house[order]], axis=0)
mdata.head()

,year,state,state_po,special,geography,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,prev_lean,prev2_lean,dem_2p_prev,dem_2p_prev2,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,urban_pct,rural_pct,demo_cluster,no_dem_cand_flag,no_rep_cand_flag,dem_inc_any,rep_inc_any,dem_inc_dummy,rep_inc_dummy,pvi,census_region,dem_scandal_score,rep_scandal_score,net_scandal_score,dem_stand,rep_stand,inc_dummy,chamber,generic_ballot_avg,polarization
0,2014,Alaska,AK,False,Alaska,129431.0,135445.0,282400.0,Mark Begich,Dan Sullivan,True,False,6020171.0,5966351.0,264876.0,48.864752,51.135248,6020171.0,5966351.00,11986522.00,50.224502,49.775498,44.935905,3.199893,44.639290,3.199893,-9.279153,-14.753632,42.684710,38.935215,68.78786,4.803705,4.887711,3.344598,13.465866,27.7,468893.0,241338.0,66.019788,33.980212,5,False,False,True,False,1,0,-10.647773,Pacific,0.0,0.0,0.0,Mark Begich,Dan Sullivan,1,Senate,2.645942,0.875383
1,2014,Arkansas,AR,False,Arkansas,334174.0,478819.0,847505.0,Mark Pryor,Tom Cotton,True,False,8000604.0,10892504.11,812993.0,41.104167,58.895833,8000604.0,10892504.11,18893108.11,42.346680,57.653320,41.723965,3.170259,47.938130,3.170259,-14.118268,-13.860568,37.845595,39.828279,79.509688,0.804425,2.788077,15.007081,0.624668,20.6,1637589.0,1278329.0,56.160324,43.839676,5,False,False,True,False,1,0,-14.053843,West South Central,0.0,0.0,0.0,Mark Pryor,Tom Cotton,1,Senate,2.645942,0.875383
2,2014,Colorado,CO,False,Colorado,944203.0,983891.0,2041058.0,Mark Udall,Cory Gardner,True,False,13414189.0,8725449.0,1928094.0,48.970797,51.029203,13414189.0,8725449.00,22139638.00,60.589017,39.410983,44.062920,9.133357,45.824609,9.133357,0.784126,0.861787,52.747988,54.550635,77.59865,2.210174,14.093498,3.831403,0.962621,37.5,4332761.0,696435.0,86.152160,13.847840,2,False,False,True,False,1,0,0.803541,Mountain,0.0,0.0,0.0,Mark Udall,Cory Gardner,1,Senate,2.645942,0.875383
3,2014,Delaware,DE,False,Delaware,130655.0,98823.0,234038.0,Chris Coons,Kevin L. Wade,True,False,2845059.0,92155.0,229478.0,56.935741,43.064259,2845059.0,92155.00,2937214.00,96.862503,3.137497,54.000000,0.469424,36.000000,0.469424,7.483093,8.949291,59.446955,62.638139,71.087114,2.115118,4.805171,20.679898,0.360503,29.4,747949.0,149985.0,83.296657,16.703343,2,False,False,True,False,1,0,7.849642,South Atlantic,0.0,0.0,0.0,Chris Coons,Kevin L. Wade,1,Senate,2.645942,0.875383
4,2014,Georgia,GA,False,Georgia,1160811.0,1358088.0,2567805.0,Michelle Nunn,David Perdue,False,False,13657065.0,8248614.0,2518899.0,46.084063,53.915937,13657065.0,8248614.00,21905679.00,62.344860,37.655140,45.051104,8.728368,47.118414,8.728368,-5.920511,-6.317183,46.043351,47.371664,61.646427,2.310356,3.83251,31.101709,0.231023,28.3,7272151.0,2415502.0,75.066180,24.933820,5,False,False,False,False,0,0,-6.019679,South Atlantic,0.0,0.0,0.0,Michelle Nunn,David Perdue,0,Senate,2.645942,0.875383


In [1203]:
mdata.shape

(2534, 58)

In [1204]:
rep_corrections = {
    ('Alaska', 'Dan Sullivan'): 'Dan S. Sullivan',
    ('Michigan', 'Mike Rogers'): 'Mike J. Rogers'
}

In [1205]:
for (cd, cand) in rep_corrections.keys():
    mask = (
        (mdata['geography'] == cd) &
        (mdata['rep_cand'] == cand)
    )

    mdata.loc[mask, 'rep_cand'] = rep_corrections[(cd, cand)]

In [1206]:
house.columns

Index(['Unnamed: 0', 'year', 'state', 'state_po', 'special', 'geography',
       'dem', 'rep', 'totalvotes', 'dem_cand', 'rep_cand', 'dem_inc',
       'rep_inc', 'dem_funds', 'rep_funds', '2party_votes', 'dem_pct_2p',
       'rep_pct_2p', 'dem_tot_funds', 'rep_tot_funds', 'tot_funds',
       'dem_funds_2p_pct', 'rep_funds_2p_pct', 'dem_poll_avg', 'dem_effn',
       'rep_poll_avg', 'rep_effn', 'seat_number', 'prev_lean', 'prev2_lean',
       'dem_2p_prev', 'dem_2p_prev2', 'generic_ballot_avg', 'ics',
       'cvap_white_pct', 'cvap_aapi_pct', 'cvap_hisp_pct', 'cvap_black_pct',
       'cvap_natam_pct', 'college', 'urban', 'rural', 'urban_pct', 'rural_pct',
       'dem_2p_12', 'demo_cluster', 'district_id', 'no_dem_cand_flag',
       'no_rep_cand_flag', 'dem_inc_any', 'rep_inc_any', 'dem_inc_dummy',
       'rep_inc_dummy', 'pvi', 'census_region', 'dem_scandal_score',
       'rep_scandal_score', 'net_scandal_score', 'Year', 'Geography',
       'dem_stand', 'rep_stand', 'inc_dummy', 'polariz

In [1211]:
mdata[(mdata['state'] == 'Maine') & (mdata['year'] == 2024)]

,year,state,state_po,special,geography,dem,rep,totalvotes,dem_cand,rep_cand,dem_inc,rep_inc,dem_funds,rep_funds,2party_votes,dem_pct_2p,rep_pct_2p,dem_tot_funds,rep_tot_funds,tot_funds,dem_funds_2p_pct,rep_funds_2p_pct,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,prev_lean,prev2_lean,dem_2p_prev,dem_2p_prev2,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,urban_pct,rural_pct,demo_cluster,no_dem_cand_flag,no_rep_cand_flag,dem_inc_any,rep_inc_any,dem_inc_dummy,rep_inc_dummy,pvi,census_region,dem_scandal_score,rep_scandal_score,net_scandal_score,dem_stand,rep_stand,inc_dummy,chamber,generic_ballot_avg,polarization
176,2024,Maine,ME,False,Maine,427570.0,284434.0,842447.0,Angus King (Ind),Demitroula Kouzounas,True,False,3734651.41,364262.81,712004.0,60.051629,39.948371,3734651.41,364262.81,4098914.22,91.113188,8.886812,51.781937,2.334136,31.801971,2.334136,2.400570,0.483502,53.544306,54.670405,94.24977,0.897865,1.689465,0.930085,0.342309,36.189622,526309.0,836050.0,38.632181,61.367819,3,False,False,True,False,1,0,1.921303,New England,0.0,0.0,0.0,Angus King (Ind),Demitroula Kouzounas,1,Senate,-0.835244,0.982821
2098,2024,Maine,ME,False,ME-01,249798.0,154849.0,425530.0,Chellie Pingree,Ronald Russell,True,False,324930.03,122597.69,404647.0,61.732325,38.267675,324930.03,122597.69,447527.72,72.605565,27.394435,57.576951,2.632691,27.528427,2.632691,9.462787,6.631204,61.041509,61.732622,93.877971,1.195485,1.835463,1.011151,0.133866,41.27,343548.0,337631.0,50.434320,49.565680,3,False,False,True,False,1,0,8.754892,New England,0.0,0.0,0.0,Chellie Pingree,Ronald Russell,1,House,-0.835244,0.982821
2099,2024,Maine,ME,False,ME-02,197151.0,194445.0,402936.0,Jared Golden,Austin Theriault,True,False,6258176.78,2273223.17,391596.0,50.345509,49.654491,6258176.78,2273223.17,8531399.95,73.354629,26.645371,47.981206,3.043343,42.639472,3.043343,-5.410176,-6.006477,45.382645,46.859659,94.623805,0.598455,1.542589,0.848532,0.552005,26.92,182761.0,498419.0,26.830060,73.169940,3,False,False,True,False,1,0,-5.559251,New England,0.0,0.0,0.0,Jared Golden,Austin Theriault,1,House,-0.835244,0.982821


In [1208]:
mdata.to_csv('transformed/all_2p_house+senate_races_trainset.csv')